# VD-EXP-002+ - BDD100K YOLO11n Vehicle Detection Fine-Tune + Specialist Comparison

Bu notebook, Anomali Road Safety AI için FTR uyumlu araç tespiti fine-tune hattını tek dosyada yürütür.

Ana karar:

- İlk resmi model: `YOLO11n`
- Ana veri omurgası: `BDD100K`
- Eğitim ortamı: Google Colab GPU
- Veri/model saklama: Google Drive
- Zorunlu çıktı: `.pt`
- Opsiyonel çıktı: ONNX
- İlk hedef: `vehicle_detector_general`
- Opsiyonel specialist profilleri: `night_low_light`, `rain`, `fog_low_visibility`

FTR notu: Bu notebook kesin saha performansı iddiası üretmez. Üretilen metrikler dataset/split/protokol bağlamında raporlanmalıdır.

## 0. Kullanım Özeti

İlk koşu için önerilen güvenli sıra:

1. `DOWNLOAD_METHOD` değerini seç.
2. İlk hücrede Google Drive mount edildiğini doğrula.
3. Bu sürümde `SMOKE_LIMIT_IMAGES = None`; dönüşüm ve eğitim tüm uygun BDD100K verisiyle çalışır.
4. `RUN_GENERAL = True` ve tüm specialist flag'leri `True`; general modelden sonra night/rain/fog specialist checkpoint'leri eğitilir.
5. Condition breakdown ve specialist-vs-general tablolarını FTR kanıtı olarak kontrol et.
6. Eğer yalnız smoke test istenirse geçici olarak `SMOKE_LIMIT_IMAGES = 500` yapılmalıdır.
7. Drive cache kullanımı varsayılan açıktır: `FORCE_REBUILD_YOLO_DATASET = False`, `FORCE_RETRAIN_MODELS = False`, `RESUME_INTERRUPTED_TRAINING = True`.

Raw dataset, model ağırlıkları ve run çıktıları Git'e eklenmez. Drive altında tutulur.

In [ ]:
# Colab setup: mount Google Drive before any /content/drive path is created.
import sys
import subprocess

try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print('Google Drive mounted at /content/drive')
except ModuleNotFoundError:
    print('Not running inside Google Colab; Drive mount skipped.')

packages = ['ultralytics', 'pyyaml', 'pandas', 'pillow', 'tqdm', 'kaggle', 'gdown', 'tabulate']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])
print('Colab dependencies installed/verified.')


Mounted at /content/drive
Google Drive mounted at /content/drive
Colab dependencies installed/verified.


## 1. Config

Notebook önce Drive'daki mevcut image verisini kullanır. Ancak label/image overlap kontrolü sıfır eşleşme bulursa, mevcut image mirror'ın label arşiviyle uyumsuz olduğunu kabul eder ve Archive.org `bdd100k_images.zip` paketini Drive'a indirip çıkararak eşleşen BDD100K 100K Images setiyle dönüşümü yeniden dener. Eksik label arşivi için ETH mirror, olmazsa Archive.org `bdd100k_labels.zip` fallback'i kullanılır.

Ana ayar:

```python
USE_EXISTING_DRIVE_DATA = True
DOWNLOAD_METHOD = 'manual'
DETECTION_LABELS_DOWNLOAD_URLS = [
    'https://dl.cv.ethz.ch/bdd100k/data/bdd100k_det_20_labels_trainval.zip',
    'https://archive.org/download/bdd100k/bdd100k_labels.zip',
]
DETECTION_LABELS_ARCHIVE_PATH = BDD_ROOT / 'bdd100k_det_20_labels_trainval.zip'
REQUIRE_DETECTION_LABELS = True
```

Bu yol BDD100K condition metadata'sini (`weather`, `timeofday`, `scene`) korur. ETH `Detection 2020 Labels` birinci tercihtir; Internet Archive fallback eski `bdd100k_labels_images_train/val.json` formatini kullanir ama condition breakdown icin yine uygundur.

Hazir YOLO Kaggle fallback'i varsayilan kapali tutulur; cunku condition metadata eksik olabilir.


In [ ]:
from pathlib import Path
from urllib.parse import urlparse
from urllib.request import urlretrieve
import collections
import csv
import hashlib
import json
import math
import os
import shutil
import subprocess
import time
import zipfile

import pandas as pd
import yaml
from PIL import Image
from tqdm.auto import tqdm

PROJECT_ROOT = Path('/content/drive/MyDrive/anomali-road-safety-ai')
# Drive is slow for extracting 100K small images. Keep archives/results on Drive,
# but extract and train from Colab local disk for the active runtime.
DRIVE_DATASET_ROOT = PROJECT_ROOT / 'datasets' / 'bdd100k'
DRIVE_OUT_ROOT = PROJECT_ROOT / 'datasets' / 'bdd100k_vehicle_yolo'
LOCAL_RUNTIME_ROOT = Path('/content/anomali-road-safety-ai-work')
USE_LOCAL_DATASET_WORKDIR = True
USE_LOCAL_YOLO_WORKDIR = True
USE_LOCAL_ARCHIVE_CACHE = True
LOCAL_ARCHIVE_CACHE = LOCAL_RUNTIME_ROOT / 'archives'
BDD_ROOT = (LOCAL_RUNTIME_ROOT / 'datasets' / 'bdd100k') if USE_LOCAL_DATASET_WORKDIR else DRIVE_DATASET_ROOT
OUT_ROOT = (LOCAL_RUNTIME_ROOT / 'datasets' / 'bdd100k_vehicle_yolo') if USE_LOCAL_YOLO_WORKDIR else DRIVE_OUT_ROOT
RUN_ROOT = PROJECT_ROOT / 'runs' / 'vehicle_detection' / 'VD-EXP-002'
SUMMARY_ROOT = RUN_ROOT / 'summary'

USE_EXISTING_DRIVE_DATA = True
DOWNLOAD_METHOD = 'manual'  # manual uses existing Drive data; kaggle/direct/gdown are fallback modes
# Kaggle fallback mirror. Use only if Drive data is missing or intentionally reset.
KAGGLE_DATASET_SLUG = 'solesensei/solesensei_bdd100k'
# Images are already on Drive. If labels are missing, add the official Detection 2020 Labels archive here.
DETECTION_LABELS_ARCHIVE_PATH = DRIVE_DATASET_ROOT / 'bdd100k_det_20_labels_trainval.zip'
DETECTION_LABELS_DOWNLOAD_URLS = [
    # Preferred: BDD100K Detection 2020 Labels.
    'https://dl.cv.ethz.ch/bdd100k/data/bdd100k_det_20_labels_trainval.zip',
    # Fallback: legacy BDD100K image labels. Keeps weather/timeofday/scene metadata.
    'https://archive.org/download/bdd100k/bdd100k_labels.zip',
]
DETECTION_LABELS_DOWNLOAD_URL = DETECTION_LABELS_DOWNLOAD_URLS[0]  # backward-compatible alias
DETECTION_LABELS_MD5_URL = 'https://dl.cv.ethz.ch/bdd100k/data/bdd100k_det_20_labels_trainval.zip.md5'
REQUIRE_DETECTION_LABELS = True
# Use if current Drive images do not match the BDD label archive. This is large (~6.5 GB).
# The notebook downloads it into Drive only after a label/image overlap failure.
AUTO_DOWNLOAD_MATCHING_BDD_IMAGES = True
MATCHING_BDD_IMAGES_ARCHIVE_PATH = DRIVE_DATASET_ROOT / 'bdd100k_images.zip'
MATCHING_BDD_IMAGES_DOWNLOAD_URLS = [
    'https://archive.org/download/bdd100k/bdd100k_images.zip',
]
MATCHING_BDD_IMAGES_DOWNLOAD_URL = MATCHING_BDD_IMAGES_DOWNLOAD_URLS[0]  # backward-compatible alias
# Archive.org bdd100k_images.zip may only provide the test split in some mirrors.
# For detection training we need the official 100K train/val image archives.
AUTO_DOWNLOAD_BDD100K_TRAIN_VAL_IMAGES = True
BDD100K_TRAIN_IMAGES_ARCHIVE_PATH = DRIVE_DATASET_ROOT / '100k_images_train.zip'
BDD100K_VAL_IMAGES_ARCHIVE_PATH = DRIVE_DATASET_ROOT / '100k_images_val.zip'
BDD100K_TRAIN_IMAGES_DOWNLOAD_URLS = [
    'https://dl.cv.ethz.ch/bdd100k/data/100k_images_train.zip',
    'http://dl.cv.ethz.ch/bdd100k/data/100k_images_train.zip',
]
BDD100K_VAL_IMAGES_DOWNLOAD_URLS = [
    'https://dl.cv.ethz.ch/bdd100k/data/100k_images_val.zip',
    'http://dl.cv.ethz.ch/bdd100k/data/100k_images_val.zip',
]
MIN_EXPECTED_TRAIN_IMAGES = 50000
MIN_EXPECTED_VAL_IMAGES = 5000
# Fallback only: ready YOLO labels can train a detector, but condition metadata may be missing.
USE_YOLO_READY_FALLBACK = False
YOLO_READY_FALLBACK_KAGGLE_SLUG = 'a7madmostafa/bdd100k-yolo'
DIRECT_URLS = []            # Optional permitted direct archives.
GDOWN_FILE_IDS = []         # Optional team Drive file ids for official archives.

TARGET_CLASSES = ['car', 'bus', 'truck', 'motorcycle']
CLASS_TO_ID = {name: idx for idx, name in enumerate(TARGET_CLASSES)}
CATEGORY_MAPPING = {
    'car': 'car',
    'bus': 'bus',
    'truck': 'truck',
    'motor': 'motorcycle',
    'motorcycle': 'motorcycle',
    'motorbike': 'motorcycle',
}
IGNORED_CATEGORIES = {
    'person', 'rider', 'bike', 'bicycle', 'traffic light', 'traffic sign',
    'train', 'other vehicle', 'trailer'
}

# İlk gerçek koşuda None yapın. Küçük smoke için örn. 500 kullanılabilir.
SMOKE_LIMIT_IMAGES = None
FORCE_REBUILD_YOLO_DATASET = False  # If False, reuse converted YOLO labels/images/metadata on Drive.
FORCE_RETRAIN_MODELS = False        # If False, reuse existing best.pt checkpoints.
RESUME_INTERRUPTED_TRAINING = True  # If best.pt is missing but last.pt exists, resume Ultralytics training.
RANDOM_SEED = 42
SPLIT_RATIOS = {'train': 0.70, 'val': 0.15, 'test': 0.15}

RUN_GENERAL = True
RUN_SPECIALISTS = True
RUN_NIGHT_SPECIALIST = True
RUN_RAIN_SPECIALIST = True
RUN_FOG_SPECIALIST = True
RUN_EXPORT_ONNX = False
RUN_LOCAL_SMOKE = False
LOCAL_SMOKE_VIDEO_DIR = PROJECT_ROOT / 'local_test_videos'  # optional; videos are not committed

IMGSZ = 640
# A100 Colab preset: use Ultralytics auto-batch as a VRAM fraction instead of a hard-coded batch.
# If Colab assigns a smaller GPU later, set HARDWARE_PROFILE='auto' to use conservative batches.
HARDWARE_PROFILE = 'a100'
DEVICE = 0
TRAIN_BATCH = 0.85 if HARDWARE_PROFILE == 'a100' else -1
VAL_BATCH = 64 if HARDWARE_PROFILE == 'a100' else 16
WORKERS = 8 if HARDWARE_PROFILE == 'a100' else 4
CACHE_MODE = 'disk' if HARDWARE_PROFILE == 'a100' else False
AMP = True
DETERMINISTIC = False if HARDWARE_PROFILE == 'a100' else True
PLOTS = True
SAVE_PERIOD = 5
EPOCHS_GENERAL = 50
EPOCHS_SPECIALIST = 30
PATIENCE = 15
MIN_TRAIN_IMAGES_FOR_EXPERIMENT = 100
MIN_VAL_IMAGES_FOR_EXPERIMENT = 20

PROFILE_CONDITIONS = {
    'general': None,
    'night_low_light': {'night_low_light', 'low_light_transition', 'tunnel_or_parking_dark'},
    'rain': {'rain'},
    'fog_low_visibility': {'fog_low_visibility'},
}

EXPERIMENTS = [
    {
        'experiment_id': 'VD-EXP-002-GENERAL-YOLO11N',
        'enabled': RUN_GENERAL,
        'profile': 'general',
        'model': 'yolo11n.pt',
        'epochs': EPOCHS_GENERAL,
        'start_from_general': False,
        'description': 'BDD100K condition-aware general detector',
    },
    {
        'experiment_id': 'VD-EXP-003-NIGHT-YOLO11N',
        'enabled': RUN_SPECIALISTS and RUN_NIGHT_SPECIALIST,
        'profile': 'night_low_light',
        'model': 'yolo11n.pt',
        'epochs': EPOCHS_SPECIALIST,
        'start_from_general': True,
        'description': 'Night/low-light specialist from best general checkpoint',
    },
    {
        'experiment_id': 'VD-EXP-004-RAIN-YOLO11N',
        'enabled': RUN_SPECIALISTS and RUN_RAIN_SPECIALIST,
        'profile': 'rain',
        'model': 'yolo11n.pt',
        'epochs': EPOCHS_SPECIALIST,
        'start_from_general': True,
        'description': 'Rain specialist from best general checkpoint',
    },
    {
        'experiment_id': 'VD-EXP-005-FOG-YOLO11N',
        'enabled': RUN_SPECIALISTS and RUN_FOG_SPECIALIST,
        'profile': 'fog_low_visibility',
        'model': 'yolo11n.pt',
        'epochs': EPOCHS_SPECIALIST,
        'start_from_general': True,
        'description': 'Fog/low-visibility specialist from best general checkpoint',
    },
]

for path in [DRIVE_DATASET_ROOT, BDD_ROOT, OUT_ROOT, RUN_ROOT, SUMMARY_ROOT, LOCAL_ARCHIVE_CACHE]:
    path.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('DRIVE_DATASET_ROOT:', DRIVE_DATASET_ROOT)
print('BDD_ROOT active workdir:', BDD_ROOT)
print('OUT_ROOT active YOLO workdir:', OUT_ROOT)
print('DRIVE_OUT_ROOT:', DRIVE_OUT_ROOT)
print('LOCAL_ARCHIVE_CACHE:', LOCAL_ARCHIVE_CACHE)
print('RUN_ROOT:', RUN_ROOT)
print('Hardware profile:', HARDWARE_PROFILE, '| device:', DEVICE, '| train_batch:', TRAIN_BATCH, '| val_batch:', VAL_BATCH, '| workers:', WORKERS, '| cache:', CACHE_MODE, '| amp:', AMP)
print('Enabled experiments:', [e['experiment_id'] for e in EXPERIMENTS if e['enabled']])


PROJECT_ROOT: /content/drive/MyDrive/anomali-road-safety-ai
DRIVE_DATASET_ROOT: /content/drive/MyDrive/anomali-road-safety-ai/datasets/bdd100k
BDD_ROOT active workdir: /content/anomali-road-safety-ai-work/datasets/bdd100k
OUT_ROOT active YOLO workdir: /content/anomali-road-safety-ai-work/datasets/bdd100k_vehicle_yolo
DRIVE_OUT_ROOT: /content/drive/MyDrive/anomali-road-safety-ai/datasets/bdd100k_vehicle_yolo
LOCAL_ARCHIVE_CACHE: /content/anomali-road-safety-ai-work/archives
RUN_ROOT: /content/drive/MyDrive/anomali-road-safety-ai/runs/vehicle_detection/VD-EXP-002
Hardware profile: a100 | device: 0 | train_batch: 0.85 | val_batch: 64 | workers: 8 | cache: disk | amp: True
Enabled experiments: ['VD-EXP-002-GENERAL-YOLO11N', 'VD-EXP-003-NIGHT-YOLO11N', 'VD-EXP-004-RAIN-YOLO11N', 'VD-EXP-005-FOG-YOLO11N']


## 2. Existing Drive Data / Optional Download

Bu hucre once Drive'da mevcut veri var mi bakar. Veri varsa hicbir sey indirmez.

Beklenen Drive kokleri:

- `/content/drive/MyDrive/anomali-road-safety-ai/datasets/bdd100k`
- `/content/drive/MyDrive/anomali-road-safety-ai/datasets/bdd100k_vehicle_yolo`

Eger veri yoksa ve `DOWNLOAD_METHOD = 'kaggle'` yapilirsa Kaggle fallback calisir.


In [ ]:
def run_cmd(cmd):
    print('+', ' '.join(map(str, cmd)))
    subprocess.check_call(list(map(str, cmd)))


def local_archive_copy(archive_path):
    archive_path = Path(archive_path)
    if not globals().get('USE_LOCAL_ARCHIVE_CACHE', False):
        return archive_path
    cache_dir = Path(globals().get('LOCAL_ARCHIVE_CACHE', '/content/archive_cache'))
    cache_dir.mkdir(parents=True, exist_ok=True)
    local_path = cache_dir / archive_path.name
    if local_path.exists() and archive_path.exists() and local_path.stat().st_size == archive_path.stat().st_size:
        print('Using cached local archive:', local_path)
        return local_path
    if not archive_path.exists():
        return archive_path
    print('Copying archive from Drive to local Colab disk for fast extraction:')
    print('  source:', archive_path)
    print('  target:', local_path)
    shutil.copy2(archive_path, local_path)
    return local_path


def archive_extract_marker(archive_path, output_dir):
    output_dir = Path(output_dir)
    safe_name = Path(archive_path).name.replace('/', '_')
    return output_dir / f'.{safe_name}.local-extracted.marker'


def extract_archive(archive_path, output_dir):
    archive_path = Path(archive_path)
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    local_archive = local_archive_copy(archive_path)
    name = archive_path.name.lower()
    if name.endswith('.zip'):
        with zipfile.ZipFile(local_archive) as zf:
            zf.extractall(output_dir)
    elif name.endswith(('.tar', '.tar.gz', '.tgz', '.tar.bz2')):
        run_cmd(['tar', '-xf', local_archive, '-C', output_dir])
    else:
        print('Archive extraction skipped for unsupported file:', archive_path)


def count_children(path, limit=20):
    path = Path(path)
    if not path.exists():
        return 0
    count = 0
    for _ in path.iterdir():
        count += 1
        if count >= limit:
            break
    return count


def drive_data_status():
    raw_count = count_children(BDD_ROOT)
    archive_count = count_children(globals().get('DRIVE_DATASET_ROOT', BDD_ROOT))
    yolo_count = count_children(OUT_ROOT)
    status = {
        'bdd_root_exists': BDD_ROOT.exists(),
        'bdd_root_children_sample_count': raw_count,
        'drive_dataset_root_exists': Path(globals().get('DRIVE_DATASET_ROOT', BDD_ROOT)).exists(),
        'drive_dataset_children_sample_count': archive_count,
        'out_root_exists': OUT_ROOT.exists(),
        'out_root_children_sample_count': yolo_count,
        'has_existing_raw_or_yolo': raw_count > 0 or archive_count > 0 or yolo_count > 0,
    }
    print('Drive dataset status:', status)
    return status


def setup_kaggle_credentials_if_needed():
    if DOWNLOAD_METHOD != 'kaggle':
        return
    username = os.getenv('KAGGLE_USERNAME')
    key = os.getenv('KAGGLE_KEY')
    if not username or not key:
        try:
            from google.colab import userdata
            username = username or userdata.get('KAGGLE_USERNAME')
            key = key or userdata.get('KAGGLE_KEY')
        except Exception:
            pass
    if not username or not key:
        import getpass
        username = input('Kaggle username: ').strip()
        key = getpass.getpass('Kaggle API key: ').strip()
    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    kaggle_json = kaggle_dir / 'kaggle.json'
    kaggle_json.write_text(json.dumps({'username': username, 'key': key}))
    os.chmod(kaggle_json, 0o600)
    print('Kaggle credentials configured at runtime.')


def download_or_verify_dataset():
    status = drive_data_status()
    if USE_EXISTING_DRIVE_DATA and status['has_existing_raw_or_yolo']:
        print('Existing Drive dataset detected. Skipping download. To force re-download, set USE_EXISTING_DRIVE_DATA = False and DOWNLOAD_METHOD = kaggle/direct/gdown.')
        return
    if DOWNLOAD_METHOD in {'manual', 'skip'}:
        raise FileNotFoundError(
            'No existing Drive dataset was detected, and DOWNLOAD_METHOD is manual/skip. '
            'Put BDD100K under BDD_ROOT or set DOWNLOAD_METHOD to kaggle/direct/gdown.'
        )
    if DOWNLOAD_METHOD == 'kaggle':
        if not KAGGLE_DATASET_SLUG:
            raise ValueError('Set KAGGLE_DATASET_SLUG before using kaggle download mode.')
        setup_kaggle_credentials_if_needed()
        run_cmd(['kaggle', 'datasets', 'download', '-d', KAGGLE_DATASET_SLUG, '-p', str(DRIVE_DATASET_ROOT)])
        for archive in DRIVE_DATASET_ROOT.glob('*'):
            if archive.suffix.lower() == '.zip' or archive.name.lower().endswith(('.tar.gz', '.tgz')):
                extract_archive(archive, BDD_ROOT)
        return
    if DOWNLOAD_METHOD == 'direct':
        if not DIRECT_URLS:
            raise ValueError('Set DIRECT_URLS before using direct download mode.')
        for url in DIRECT_URLS:
            filename = Path(urlparse(url).path).name or 'downloaded_archive'
            target = DRIVE_DATASET_ROOT / filename
            print('Downloading:', url)
            urlretrieve(url, target)
            extract_archive(target, BDD_ROOT)
        return
    if DOWNLOAD_METHOD == 'gdown':
        if not GDOWN_FILE_IDS:
            raise ValueError('Set GDOWN_FILE_IDS before using gdown mode.')
        for file_id in GDOWN_FILE_IDS:
            run_cmd(['gdown', '--id', file_id, '-O', str(DRIVE_DATASET_ROOT / f'{file_id}.download')])
        for archive in DRIVE_DATASET_ROOT.glob('*.download'):
            extract_archive(archive, BDD_ROOT)
        return
    raise ValueError('Unknown DOWNLOAD_METHOD: ' + DOWNLOAD_METHOD)


download_or_verify_dataset()

Drive dataset status: {'bdd_root_exists': True, 'bdd_root_children_sample_count': 0, 'drive_dataset_root_exists': True, 'drive_dataset_children_sample_count': 6, 'out_root_exists': True, 'out_root_children_sample_count': 0, 'has_existing_raw_or_yolo': True}
Existing Drive dataset detected. Skipping download. To force re-download, set USE_EXISTING_DRIVE_DATA = False and DOWNLOAD_METHOD = kaggle/direct/gdown.


## 3. BDD100K Path Discovery + Repair

Kaggle mirror klasor yapisi resmi BDD100K path'leriyle birebir ayni olmayabilir. Bu hucre nested archive'leri acar, klasor agacini kisa gosterir ve label/image path'lerini recursive olarak bulur.

Eger burada label JSON bulunamazsa indirdigimiz mirror object detection label dosyalarini icermiyor demektir; o durumda Kaggle kaynagini degistirmek veya resmi Detection 2020 Labels arsivini Drive'a eklemek gerekir.


In [ ]:
import tarfile
import zipfile
from itertools import islice


def quick_existing_image_dirs(root):
    root = Path(root)
    candidates = [
        root / 'images' / '100k' / 'train',
        root / 'bdd100k' / 'images' / '100k' / 'train',
        root / 'bdd100k' / 'bdd100k' / 'images' / '100k' / 'train',
        root / '100k' / 'train',
        root / 'images' / 'train',
        root / 'train',
    ]
    return [p for p in candidates if p.exists() and p.is_dir()]


def quick_json_candidates(root):
    root = Path(root)
    direct = [
        root / 'labels' / 'bdd100k_labels_images_train.json',
        root / 'labels' / 'det_20' / 'det_train.json',
        root / 'bdd100k' / 'labels' / 'bdd100k_labels_images_train.json',
        root / 'bdd100k' / 'labels' / 'det_20' / 'det_train.json',
        root / 'bdd100k' / 'bdd100k' / 'labels' / 'bdd100k_labels_images_train.json',
        root / 'bdd100k_labels_images_train.json',
    ]
    per_image_dirs = [
        root / 'labels' / '100k' / 'train',
        root / 'bdd100k' / 'labels' / '100k' / 'train',
        root / 'bdd100k' / 'bdd100k' / 'labels' / '100k' / 'train',
    ]
    found = [p for p in direct if p.exists()]
    found.extend(p for p in per_image_dirs if p.exists() and p.is_dir() and next(p.glob('*.json'), None) is not None)
    return found


def maybe_extract_nested_archives(root, max_rounds=2):
    root = Path(root)
    if quick_existing_image_dirs(root) and quick_json_candidates(root):
        print('Images and train label JSON already exist. Skipping archive extraction.')
        return
    if quick_existing_image_dirs(root) and not quick_json_candidates(root):
        print('Images already exist but train label JSON was not found in common paths.')
        print('Skipping large image archive extraction. Will search JSON files recursively next.')
        return

    extracted_marker_suffix = '.extracted.marker'
    for round_idx in range(max_rounds):
        archives = []
        for pattern in ('*.zip', '*.tar', '*.tar.gz', '*.tgz'):
            archives.extend(root.glob(pattern))
            archives.extend((root / 'bdd100k').glob(pattern) if (root / 'bdd100k').exists() else [])
        pending = [a for a in archives if not (a.parent / (a.name + extracted_marker_suffix)).exists()]
        if not pending:
            break
        print(f'Archive scan round {round_idx + 1}: extracting {len(pending)} top-level archives')
        for archive in pending:
            try:
                out_dir = archive.parent / archive.stem.replace('.tar', '')
                out_dir.mkdir(parents=True, exist_ok=True)
                print('extract:', archive, '->', out_dir)
                if archive.name.endswith('.zip'):
                    with zipfile.ZipFile(archive) as zf:
                        zf.extractall(out_dir)
                elif archive.name.endswith(('.tar', '.tar.gz', '.tgz')):
                    with tarfile.open(archive) as tf:
                        tf.extractall(out_dir)
                (archive.parent / (archive.name + extracted_marker_suffix)).write_text('ok')
            except Exception as exc:
                print('archive extract failed:', archive, exc)


def list_tree_preview(root, max_items=120, max_depth=4):
    root = Path(root)
    print('Tree preview under:', root)
    shown = 0
    stack = [(root, 0)]
    while stack and shown < max_items:
        current, depth = stack.pop(0)
        if depth > 0:
            try:
                rel = current.relative_to(root)
            except Exception:
                rel = current
            print(str(rel) + ('/' if current.is_dir() else ''))
            shown += 1
        if current.is_dir() and depth < max_depth:
            try:
                children = list(islice(current.iterdir(), 60))
            except Exception:
                children = []
            # Show folders first without traversing the whole 100k image set.
            children = sorted(children, key=lambda p: (not p.is_dir(), p.name.lower()))
            stack.extend((child, depth + 1) for child in children)
    if shown >= max_items:
        print('... truncated')


def json_score(path, split):
    name = path.name.lower()
    full = str(path).lower()
    score = 0
    if split in name:
        score += 10
    if 'det_' in full or 'detect' in full or 'detection' in full:
        score += 8
    if 'label' in full or 'labels' in full:
        score += 6
    if 'image' in full or 'images' in full or '100k' in full:
        score += 2
    if name in {f'det_{split}.json', f'bdd100k_labels_images_{split}.json'}:
        score += 20
    return score


def label_dir_candidates(root, split):
    root = Path(root)
    candidates = [
        root / 'labels' / '100k' / split,
        root / 'bdd100k' / 'labels' / '100k' / split,
        root / 'bdd100k' / 'bdd100k' / 'labels' / '100k' / split,
        root / 'labels' / split,
        root / 'bdd100k' / 'labels' / split,
    ]
    return [p for p in candidates if p.exists() and p.is_dir() and next(p.glob('*.json'), None) is not None]


def looks_like_bdd_entry(obj):
    return isinstance(obj, dict) and ('name' in obj) and any(
        key in obj for key in ('labels', 'frames', 'objects', 'annotations', 'attributes')
    )

def iter_json_candidates_fast(root, split):
    root = Path(root)
    common = [
        root / 'labels' / f'bdd100k_labels_images_{split}.json',
        root / 'labels' / 'det_20' / f'det_{split}.json',
        root / 'bdd100k' / 'labels' / f'bdd100k_labels_images_{split}.json',
        root / 'bdd100k' / 'labels' / 'det_20' / f'det_{split}.json',
        root / 'bdd100k' / 'bdd100k' / 'labels' / f'bdd100k_labels_images_{split}.json',
        root / f'bdd100k_labels_images_{split}.json',
        root / f'det_{split}.json',
    ]
    seen = set()
    for path in common:
        if path.exists() and path not in seen:
            seen.add(path)
            yield path
    # Search JSON files only; avoid image tree preview/full sorting.
    for path in root.rglob('*.json'):
        if path in seen:
            continue
        full = str(path).lower()
        if split in full and any(token in full for token in ['label', 'det_', 'detect', 'detection', 'bdd100k']):
            seen.add(path)
            yield path


def find_label_json(split):
    # Prefer official aggregate JSON files when present.
    candidates = sorted(iter_json_candidates_fast(BDD_ROOT, split), key=lambda p: json_score(p, split), reverse=True)
    aggregate_candidates = []
    individual_candidates = []
    for path in candidates[:80]:
        try:
            with open(path, 'r') as f:
                sample = json.load(f)
            if isinstance(sample, list) and sample:
                first = sample[0]
                if looks_like_bdd_entry(first):
                    aggregate_candidates.append(path)
                    continue
            if looks_like_bdd_entry(sample):
                individual_candidates.append(path)
                continue
            if isinstance(sample, dict) and ('frames' in sample or 'labels' in sample or 'annotations' in sample):
                aggregate_candidates.append(path)
                continue
        except Exception as exc:
            print('skip json candidate:', path, exc)
    if aggregate_candidates:
        print(f'{split.upper()}_LABEL ->', aggregate_candidates[0])
        return aggregate_candidates[0]

    # Archive.org fallback extracts per-image JSON files under labels/100k/{train,val}/.
    label_dirs = label_dir_candidates(BDD_ROOT, split)
    if label_dirs:
        print(f'{split.upper()}_LABEL_DIR ->', label_dirs[0])
        return label_dirs[0]

    if individual_candidates:
        parent = individual_candidates[0].parent
        if parent.exists() and parent.is_dir():
            print(f'{split.upper()}_LABEL_DIR ->', parent)
            return parent

    print('JSON candidates found but none matched BDD detection label structure:')
    for path in candidates[:30]:
        print(' ', path)
    raise FileNotFoundError(
        f'{split} label source not found under {BDD_ROOT}. '
        'Expected aggregate det_train/det_val JSON or per-image labels/100k/{train,val}/*.json.'
    )

def image_dir_score(path, split):
    full = str(path).lower()
    score = 0
    if split in full:
        score += 10
    if '100k' in full:
        score += 5
    if 'image' in full or 'images' in full:
        score += 5
    return score


def has_any_image(path):
    path = Path(path)
    for ext in ('*.jpg', '*.jpeg', '*.png'):
        try:
            if next(path.glob(ext), None) is not None:
                return True
        except Exception:
            pass
    return False


def find_image_dir(split, required=True):
    common_candidates = []
    for path in bdd_split_dir_candidates(split):
        if path.exists() and path.is_dir():
            count = count_images_recursive(path)
            if count:
                common_candidates.append((count, image_dir_score(path, split), path))
    if common_candidates:
        common_candidates.sort(key=lambda item: (item[0], item[1]), reverse=True)
        count, _, path = common_candidates[0]
        print(f'{split.upper()}_IMAGE_DIR ->', path, 'images:', count)
        return path

    dirs = []
    for d in BDD_ROOT.rglob(split):
        if d.is_dir() and has_any_image(d):
            count = count_images_recursive(d)
            dirs.append((count, image_dir_score(d, split), d))
    if not dirs:
        if required:
            list_tree_preview(BDD_ROOT, max_items=160)
            raise FileNotFoundError(f'{split} image directory not found under {BDD_ROOT}')
        print(f'{split.upper()}_IMAGE_DIR -> not found. This is allowed; converted train images will be hash-split into train/val/test.')
        return None
    dirs = sorted(dirs, key=lambda item: (item[0], item[1]), reverse=True)
    print(f'{split.upper()}_IMAGE_DIR ->', dirs[0][2], 'images:', dirs[0][0])
    return dirs[0][2]


def sanitize_download_url(value):
    url = str(value).strip()
    # Some notebook/chat copy paths turn URLs into Markdown links: [https://x](https://x).
    if url.startswith('[') and '](' in url and url.endswith(')'):
        url = url.split('](', 1)[1][:-1].strip()
    if url.startswith('<') and url.endswith('>'):
        url = url[1:-1].strip()
    return url

def download_file_with_fallback(urls, target_path, min_size_mb=1, artifact_name='archive'):
    target_path = Path(target_path)
    target_path.parent.mkdir(parents=True, exist_ok=True)
    if isinstance(urls, str):
        urls = [urls]
    errors = []
    for raw_url in urls:
        url = sanitize_download_url(raw_url)
        if url != str(raw_url).strip():
            print('Sanitized download URL:', raw_url, '->', url)
        print(f'Trying {artifact_name} URL:', url)
        # 1) urllib: no shell dependency.
        try:
            tmp = target_path.with_suffix(target_path.suffix + '.part')
            if tmp.exists():
                tmp.unlink()
            urlretrieve(url, tmp)
            if tmp.stat().st_size < min_size_mb * 1024 * 1024:
                raise RuntimeError(f'downloaded file too small: {tmp.stat().st_size} bytes')
            tmp.replace(target_path)
            print('Downloaded via urllib:', target_path)
            return target_path
        except Exception as exc:
            errors.append(f'urllib failed for {url}: {exc}')
            print(errors[-1])

        # 2) curl: often more robust on Colab transient DNS/TLS issues.
        try:
            tmp = target_path.with_suffix(target_path.suffix + '.part')
            if tmp.exists():
                tmp.unlink()
            run_cmd(['curl', '-L', '--retry', '5', '--retry-delay', '5', '--connect-timeout', '30', '-o', tmp, url])
            if tmp.stat().st_size < min_size_mb * 1024 * 1024:
                raise RuntimeError(f'downloaded file too small: {tmp.stat().st_size} bytes')
            tmp.replace(target_path)
            print('Downloaded via curl:', target_path)
            return target_path
        except Exception as exc:
            errors.append(f'curl failed for {url}: {exc}')
            print(errors[-1])

        # 3) wget fallback.
        try:
            tmp = target_path.with_suffix(target_path.suffix + '.part')
            if tmp.exists():
                tmp.unlink()
            run_cmd(['wget', '-O', tmp, '--tries=5', '--timeout=30', url])
            if tmp.stat().st_size < min_size_mb * 1024 * 1024:
                raise RuntimeError(f'downloaded file too small: {tmp.stat().st_size} bytes')
            tmp.replace(target_path)
            print('Downloaded via wget:', target_path)
            return target_path
        except Exception as exc:
            errors.append(f'wget failed for {url}: {exc}')
            print(errors[-1])

    raise RuntimeError(
        f'Could not download {artifact_name} from any configured URL.\n'
        'If Colab DNS/network is blocked, manually place this file at:\n'
        f'  {target_path}\n'
        'Errors:\n- ' + '\n- '.join(errors)
    )


def maybe_download_detection_labels_archive():
    archive = Path(DETECTION_LABELS_ARCHIVE_PATH)
    if archive.exists() and archive.stat().st_size > 1024 * 1024:
        print('Detection labels archive already exists:', archive)
        return
    urls = globals().get('DETECTION_LABELS_DOWNLOAD_URLS', None)
    if not urls:
        legacy_url = globals().get('DETECTION_LABELS_DOWNLOAD_URL', '')
        urls = [legacy_url] if legacy_url else []
    if not urls:
        print('No label archive download URL configured; label archive download skipped.')
        return
    print('Downloading Detection 2020 Labels only. Images will not be re-downloaded.')
    print(' ->', archive)
    download_file_with_fallback(urls, archive, min_size_mb=10, artifact_name='BDD100K label archive')
    print('Downloaded labels archive size MB:', round(archive.stat().st_size / (1024 * 1024), 2))

def maybe_extract_detection_labels_archive():
    archive = Path(DETECTION_LABELS_ARCHIVE_PATH)
    common_labels_exist = bool(quick_json_candidates(BDD_ROOT))
    if common_labels_exist:
        print('Detection label JSON already exists in common paths. Skipping label archive extraction.')
        return
    if not archive.exists():
        maybe_download_detection_labels_archive()
    if not archive.exists():
        print('Detection labels archive not found at:', archive)
        print('Add BDD100K Detection 2020 Labels or set DETECTION_LABELS_DOWNLOAD_URL.')
        return
    out_dir = BDD_ROOT
    marker = archive_extract_marker(archive, out_dir)
    if marker.exists() and quick_json_candidates(BDD_ROOT):
        print('Detection labels archive was already extracted into active workdir:', archive)
        return
    print('Extracting Detection 2020 Labels archive into active workdir:', archive, '->', out_dir)
    try:
        local_archive = local_archive_copy(archive)
        if archive.name.endswith('.zip'):
            with zipfile.ZipFile(local_archive) as zf:
                zf.extractall(out_dir)
        elif archive.name.endswith(('.tar', '.tar.gz', '.tgz')):
            with tarfile.open(local_archive) as tf:
                tf.extractall(out_dir)
        else:
            print('Unsupported labels archive type:', archive)
            return
        marker.write_text('ok')
    except Exception as exc:
        print('Detection labels extraction failed:', exc)
        raise



def count_images_recursive(root):
    root = Path(root)
    if not root.exists() or not root.is_dir():
        return 0
    total = 0
    for ext in ('*.jpg', '*.jpeg', '*.png'):
        total += sum(1 for _ in root.rglob(ext))
    return total


def bdd_split_dir_candidates(split):
    return [
        BDD_ROOT / 'images' / '100k' / split,
        BDD_ROOT / 'bdd100k' / 'images' / '100k' / split,
        BDD_ROOT / 'bdd100k' / 'bdd100k' / 'images' / '100k' / split,
        BDD_ROOT / '100k' / split,
        BDD_ROOT / 'images' / split,
        BDD_ROOT / split,
    ]


def best_existing_split_dir(split):
    candidates = []
    for path in bdd_split_dir_candidates(split):
        if path.exists() and path.is_dir():
            count = count_images_recursive(path)
            if count:
                candidates.append((count, path))
    if not candidates:
        return None, 0
    candidates.sort(key=lambda item: item[0], reverse=True)
    return candidates[0][1], candidates[0][0]


def split_has_enough_images(split, min_count):
    path, count = best_existing_split_dir(split)
    if path is not None:
        print(f'Best existing BDD100K {split} image dir:', path, 'images:', count)
    return path is not None and count >= min_count


def download_and_extract_image_archive(urls, archive_path, split_name, min_size_mb=1000):
    archive = Path(archive_path)
    if archive.exists() and archive.stat().st_size > min_size_mb * 1024 * 1024:
        print(f'{split_name} image archive already exists:', archive)
    else:
        print(f'Downloading official BDD100K {split_name} images archive into Drive.')
        print(' ->', archive)
        download_file_with_fallback(urls, archive, min_size_mb=min_size_mb, artifact_name=f'BDD100K {split_name} images archive')
    marker = archive_extract_marker(archive, BDD_ROOT)
    if marker.exists() and count_split_images('train') >= 1:
        print(f'{split_name} image archive already extracted into active workdir:', archive)
        return
    print(f'Extracting official BDD100K {split_name} images archive into active workdir:', BDD_ROOT)
    local_archive = local_archive_copy(archive)
    if archive.name.endswith('.zip'):
        with zipfile.ZipFile(local_archive) as zf:
            zf.extractall(BDD_ROOT)
    elif archive.name.endswith(('.tar', '.tar.gz', '.tgz')):
        with tarfile.open(local_archive) as tf:
            tf.extractall(BDD_ROOT)
    else:
        raise RuntimeError(f'Unsupported {split_name} image archive type: {archive}')
    marker.write_text('ok')



def inspect_image_archive_splits(archive_path, max_preview=6):
    archive = Path(archive_path)
    counts = collections.Counter()
    preview = []
    if not archive.exists():
        return counts
    print('Inspecting existing image archive split contents:', archive)
    try:
        with zipfile.ZipFile(archive) as zf:
            for info in zf.infolist():
                name = info.filename
                lower = name.lower()
                if not lower.endswith(('.jpg', '.jpeg', '.png')):
                    continue
                for split in ('train', 'val', 'test'):
                    if f'/100k/{split}/' in lower or f'images/{split}/' in lower:
                        counts[split] += 1
                        if len(preview) < max_preview:
                            preview.append(name)
                        break
    except Exception as exc:
        print('Could not inspect archive:', archive, exc)
        return collections.Counter()
    print('Archive split counts:', dict(counts))
    if preview:
        print('Archive preview:', preview)
    return counts


def try_reextract_existing_combined_image_archive():
    archive = Path(globals().get('MATCHING_BDD_IMAGES_ARCHIVE_PATH', BDD_ROOT / 'bdd100k_images.zip'))
    if not archive.exists():
        return False
    counts = inspect_image_archive_splits(archive)
    if counts.get('train', 0) < MIN_EXPECTED_TRAIN_IMAGES or counts.get('val', 0) < MIN_EXPECTED_VAL_IMAGES:
        print('Existing combined image archive does not appear to contain complete train/val splits; skipping re-extract.')
        return False
    marker = archive_extract_marker(archive, BDD_ROOT)
    if marker.exists():
        marker.unlink()
        print('Removed combined image archive marker to force local re-extract:', marker)
    maybe_extract_matching_bdd_images_archive()
    return True


def ensure_official_train_val_images():
    if not globals().get('AUTO_DOWNLOAD_BDD100K_TRAIN_VAL_IMAGES', False):
        print('AUTO_DOWNLOAD_BDD100K_TRAIN_VAL_IMAGES is False; train/val image auto-download skipped.')
        return
    train_ok = split_has_enough_images('train', MIN_EXPECTED_TRAIN_IMAGES)
    val_ok = split_has_enough_images('val', MIN_EXPECTED_VAL_IMAGES)
    if train_ok and val_ok:
        print('BDD100K train/val image splits look complete enough. No train/val image download needed.')
        return
    print('BDD100K train/val splits are missing or incomplete. Checking existing combined image archive before network download.')
    if try_reextract_existing_combined_image_archive():
        train_ok = split_has_enough_images('train', MIN_EXPECTED_TRAIN_IMAGES)
        val_ok = split_has_enough_images('val', MIN_EXPECTED_VAL_IMAGES)
        if train_ok and val_ok:
            print('BDD100K train/val image splits completed by re-extracting existing combined archive.')
            return
    print('Downloading official train/val image archives.')
    if not train_ok:
        download_and_extract_image_archive(
            BDD100K_TRAIN_IMAGES_DOWNLOAD_URLS,
            BDD100K_TRAIN_IMAGES_ARCHIVE_PATH,
            'train',
            min_size_mb=1000,
        )
    if not val_ok:
        download_and_extract_image_archive(
            BDD100K_VAL_IMAGES_DOWNLOAD_URLS,
            BDD100K_VAL_IMAGES_ARCHIVE_PATH,
            'val',
            min_size_mb=200,
        )
    train_ok = split_has_enough_images('train', MIN_EXPECTED_TRAIN_IMAGES)
    val_ok = split_has_enough_images('val', MIN_EXPECTED_VAL_IMAGES)
    if not (train_ok and val_ok):
        list_tree_preview(BDD_ROOT, max_items=160)
        raise RuntimeError(
            'BDD100K train/val images are still incomplete after attempted download/extraction. '
            'Expected about 70000 train and 10000 val images. '
            'Do not run YOLO conversion until these splits are present.'
        )


def maybe_download_matching_bdd_images_archive():
    archive = Path(globals().get('MATCHING_BDD_IMAGES_ARCHIVE_PATH', BDD_ROOT / 'bdd100k_images.zip'))
    if archive.exists() and archive.stat().st_size > 1024 * 1024 * 1024:
        print('Matching BDD100K image archive already exists:', archive)
        return archive
    urls = globals().get('MATCHING_BDD_IMAGES_DOWNLOAD_URLS', None)
    if not urls:
        legacy_url = globals().get('MATCHING_BDD_IMAGES_DOWNLOAD_URL', '')
        urls = [legacy_url] if legacy_url else []
    if not urls:
        raise RuntimeError('No matching BDD100K image archive URL configured.')
    print('Downloading matching BDD100K 100K Images archive into Drive. This is large (~6.5 GB).')
    print(' ->', archive)
    download_file_with_fallback(urls, archive, min_size_mb=1000, artifact_name='BDD100K matching image archive')
    print('Downloaded matching image archive size GB:', round(archive.stat().st_size / (1024 ** 3), 2))
    return archive


def maybe_extract_matching_bdd_images_archive():
    if not globals().get('AUTO_DOWNLOAD_MATCHING_BDD_IMAGES', False):
        print('AUTO_DOWNLOAD_MATCHING_BDD_IMAGES is False; matching image archive download skipped.')
        return False
    archive = Path(globals().get('MATCHING_BDD_IMAGES_ARCHIVE_PATH', BDD_ROOT / 'bdd100k_images.zip'))
    if not archive.exists():
        archive = maybe_download_matching_bdd_images_archive()
    marker = archive_extract_marker(archive, BDD_ROOT)
    if marker.exists() and count_split_images('train') >= MIN_EXPECTED_TRAIN_IMAGES and count_split_images('val') >= MIN_EXPECTED_VAL_IMAGES:
        print('Matching BDD100K image archive was already extracted into active workdir:', archive)
        return True
    print('Extracting matching BDD100K image archive into active workdir:', BDD_ROOT)
    local_archive = local_archive_copy(archive)
    if archive.name.endswith('.zip'):
        with zipfile.ZipFile(local_archive) as zf:
            zf.extractall(BDD_ROOT)
    elif archive.name.endswith(('.tar', '.tar.gz', '.tgz')):
        with tarfile.open(local_archive) as tf:
            tf.extractall(BDD_ROOT)
    else:
        raise RuntimeError(f'Unsupported matching image archive type: {archive}')
    marker.write_text('ok')
    print('Matching BDD100K image archive extracted into active workdir:', archive)
    return True


print('Active BDD workdir:', BDD_ROOT)
print('Archive source root:', globals().get('DRIVE_DATASET_ROOT', BDD_ROOT))
if globals().get('USE_LOCAL_DATASET_WORKDIR', False):
    print('Fast mode enabled: archives stay on Drive, extracted images/labels go to local /content workdir.')
maybe_extract_detection_labels_archive()
maybe_extract_nested_archives(BDD_ROOT)
ensure_official_train_val_images()
list_tree_preview(BDD_ROOT, max_items=80)

TRAIN_LABEL = find_label_json('train')
try:
    VAL_LABEL = find_label_json('val')
except FileNotFoundError as exc:
    print('VAL_LABEL -> not found. Continuing with train labels only:', exc)
    VAL_LABEL = None
TRAIN_IMAGE_DIR = find_image_dir('train')
VAL_IMAGE_DIR = find_image_dir('val', required=False)

print('\nResolved BDD100K paths:')
print('TRAIN_LABEL =', TRAIN_LABEL)
print('VAL_LABEL =', VAL_LABEL)
print('TRAIN_IMAGE_DIR =', TRAIN_IMAGE_DIR)
print('VAL_IMAGE_DIR =', VAL_IMAGE_DIR)


Active BDD workdir: /content/anomali-road-safety-ai-work/datasets/bdd100k
Archive source root: /content/drive/MyDrive/anomali-road-safety-ai/datasets/bdd100k
Fast mode enabled: archives stay on Drive, extracted images/labels go to local /content workdir.
Extracting Detection 2020 Labels archive into active workdir: /content/drive/MyDrive/anomali-road-safety-ai/datasets/bdd100k/bdd100k_det_20_labels_trainval.zip -> /content/anomali-road-safety-ai-work/datasets/bdd100k
Copying archive from Drive to local Colab disk for fast extraction:
  source: /content/drive/MyDrive/anomali-road-safety-ai/datasets/bdd100k/bdd100k_det_20_labels_trainval.zip
  target: /content/anomali-road-safety-ai-work/archives/bdd100k_det_20_labels_trainval.zip
BDD100K train/val splits are missing or incomplete. Checking existing combined image archive before network download.
Inspecting existing image archive split contents: /content/drive/MyDrive/anomali-road-safety-ai/datasets/bdd100k/bdd100k_images.zip
Archive spl

## 4. Condition Mapping ve Split Fonksiyonları

Condition metadata araç sınıfı değildir. Eğitim filtreleme, balancing, validation breakdown ve specialist kararları için tutulur.

In [ ]:
def normalize_text(value):
    if value is None:
        return 'undefined'
    return str(value).strip().lower()


def condition_from_attributes(attrs):
    attrs = attrs or {}
    weather = normalize_text(attrs.get('weather'))
    timeofday = normalize_text(attrs.get('timeofday'))
    scene = normalize_text(attrs.get('scene'))
    candidates = []
    if weather in {'foggy', 'fog'}:
        candidates.append('fog_low_visibility')
    if weather in {'rainy', 'rain'}:
        candidates.append('rain')
    if timeofday in {'night'}:
        candidates.append('night_low_light')
    if timeofday in {'dawn/dusk', 'dawn', 'dusk'}:
        candidates.append('low_light_transition')
    if weather in {'snowy', 'snow', 'sandstorm'}:
        candidates.append('adverse_other')
    if scene in {'tunnel', 'parking lot'}:
        candidates.append('tunnel_or_parking_dark')
    if not candidates and (timeofday in {'daytime', 'day'} or weather in {'clear', 'overcast', 'partly cloudy'}):
        candidates.append('day_clear')
    if not candidates:
        candidates.append('unknown')
    priority = ['fog_low_visibility', 'rain', 'night_low_light', 'low_light_transition', 'tunnel_or_parking_dark', 'adverse_other', 'day_clear', 'unknown']
    return sorted(candidates, key=lambda item: priority.index(item) if item in priority else 999)[0]


def stable_bucket(value):
    digest = hashlib.md5(str(value).encode('utf-8')).hexdigest()
    return int(digest[:8], 16) / 0xFFFFFFFF


def split_for_name(name):
    b = stable_bucket(name)
    if b < SPLIT_RATIOS['train']:
        return 'train'
    if b < SPLIT_RATIOS['train'] + SPLIT_RATIOS['val']:
        return 'val'
    return 'test'


def profile_accepts_condition(profile, condition):
    allowed = PROFILE_CONDITIONS.get(profile)
    return allowed is None or condition in allowed

## 4.5 Colab/Drive Cache Repair

Bu hücre Drive üzerinde kalmış boş/bozuk conversion artifactlerini temizler. `train_label_entries_cache.json` korunur; böylece 23K label JSON yeniden okunmaz. Sağlam `bdd100k_vehicle_metadata.csv` varsa hiçbir şeye dokunmaz.


In [ ]:
# Run-safe cache repair before YOLO conversion.
# Keeps expensive label caches and raw/downloaded BDD100K archives; removes only stale conversion outputs.

def metadata_csv_is_valid_for_reuse(path):
    path = Path(path)
    required = {
        'image_name', 'image_path', 'label_path', 'source_split', 'split',
        'condition_profile', 'bbox_count', *[f'{cls}_count' for cls in TARGET_CLASSES]
    }
    if not path.exists():
        return False, 'metadata file does not exist'
    try:
        df = pd.read_csv(path)
    except Exception as exc:
        return False, f'metadata csv could not be read: {exc}'
    if df.empty:
        return False, 'metadata csv is empty'
    missing = sorted(required - set(df.columns))
    if missing:
        return False, 'metadata csv is missing columns: ' + ', '.join(missing)
    return True, f'metadata csv is valid with {len(df)} rows'


def repair_stale_conversion_artifacts():
    metadata_path = OUT_ROOT / 'metadata' / 'bdd100k_vehicle_metadata.csv'
    valid, reason = metadata_csv_is_valid_for_reuse(metadata_path)
    print('Conversion metadata status:', reason)
    if valid:
        print('Valid YOLO conversion metadata exists; preserving image indexes and partial files.')
        return

    stale_paths = [
        OUT_ROOT / 'metadata' / 'train_image_index.csv',
        OUT_ROOT / 'metadata' / 'val_image_index.csv',
        OUT_ROOT / 'metadata' / 'bdd100k_vehicle_metadata.csv',
        OUT_ROOT / 'metadata' / 'train_conversion_rows_partial.csv',
        OUT_ROOT / 'metadata' / 'val_conversion_rows_partial.csv',
    ]
    for path in stale_paths:
        if path.exists():
            path.unlink()
            print('deleted stale conversion artifact:', path)
    print('Preserved expensive label cache if present:', OUT_ROOT / 'metadata' / 'train_label_entries_cache.json')
    print('Preserved raw/downloaded BDD100K images and archives under:', BDD_ROOT)

repair_stale_conversion_artifacts()


Conversion metadata status: metadata file does not exist
Preserved expensive label cache if present: /content/anomali-road-safety-ai-work/datasets/bdd100k_vehicle_yolo/metadata/train_label_entries_cache.json
Preserved raw/downloaded BDD100K images and archives under: /content/anomali-road-safety-ai-work/datasets/bdd100k


## 5. BDD100K -> YOLO Conversion

Bu hücre BDD100K JSON label dosyalarını YOLO formatına çevirir, image symlink/copy oluşturur, split ve condition metadata CSV üretir.

In [ ]:
def read_json(path):
    with open(path, 'r') as f:
        return json.load(f)


def candidate_label_lists(obj):
    lists = []
    if not isinstance(obj, dict):
        return lists
    for key in ('labels', 'objects', 'annotations'):
        value = obj.get(key)
        if isinstance(value, list):
            lists.append(value)
    frames = obj.get('frames')
    if isinstance(frames, list):
        for frame in frames:
            if not isinstance(frame, dict):
                continue
            for key in ('labels', 'objects', 'annotations'):
                value = frame.get(key)
                if isinstance(value, list):
                    lists.append(value)
    return lists


def iter_entry_labels(entry):
    seen = set()
    for values in candidate_label_lists(entry):
        for obj in values:
            if not isinstance(obj, dict):
                continue
            # Avoid double counting when a normalized entry preserves both root and frame lists.
            key = (
                obj.get('id'),
                obj.get('category') or obj.get('class') or obj.get('label') or obj.get('name'),
                json.dumps(obj.get('box2d') or obj.get('bbox') or obj.get('box'), sort_keys=True, default=str),
            )
            if key in seen:
                continue
            seen.add(key)
            yield obj


def extract_category(obj):
    for key in ('category', 'class', 'label', 'name'):
        value = obj.get(key)
        if value:
            return normalize_text(value)
    return ''


def extract_box2d(obj):
    box = obj.get('box2d') or obj.get('bbox') or obj.get('box')
    if isinstance(box, dict):
        if all(key in box for key in ('x1', 'y1', 'x2', 'y2')):
            return box
        if all(key in box for key in ('x', 'y', 'w', 'h')):
            x = float(box['x'])
            y = float(box['y'])
            return {'x1': x, 'y1': y, 'x2': x + float(box['w']), 'y2': y + float(box['h'])}
        if all(key in box for key in ('left', 'top', 'right', 'bottom')):
            return {'x1': box['left'], 'y1': box['top'], 'x2': box['right'], 'y2': box['bottom']}
    if isinstance(box, (list, tuple)) and len(box) == 4:
        x1, y1, x2, y2 = [float(v) for v in box]
        if x2 <= x1 or y2 <= y1:
            x2 = x1 + max(0.0, x2)
            y2 = y1 + max(0.0, y2)
        return {'x1': x1, 'y1': y1, 'x2': x2, 'y2': y2}
    return None


def object_is_target_vehicle(obj):
    mapped = CATEGORY_MAPPING.get(extract_category(obj))
    return mapped in CLASS_TO_ID and extract_box2d(obj) is not None


def entry_target_box_count(entry):
    return sum(1 for obj in iter_entry_labels(entry) if object_is_target_vehicle(obj))


def normalize_bdd_entry(entry, fallback_name=None):
    if not isinstance(entry, dict):
        return None
    normalized = dict(entry)
    frames = entry.get('frames')
    frame = frames[0] if isinstance(frames, list) and frames and isinstance(frames[0], dict) else {}
    normalized.setdefault('name', entry.get('name') or frame.get('name') or fallback_name)
    attrs = {}
    if isinstance(entry.get('attributes'), dict):
        attrs.update(entry.get('attributes'))
    if isinstance(frame.get('attributes'), dict):
        attrs.update(frame.get('attributes'))
    normalized['attributes'] = attrs
    labels = list(iter_entry_labels(normalized))
    if labels:
        normalized['labels'] = labels
    return normalized if looks_like_bdd_entry(normalized) else None


def summarize_label_entries(entries, split, sample_size=1000):
    total_entries = min(len(entries), sample_size)
    label_counter = collections.Counter()
    target_boxes = 0
    entries_with_targets = 0
    sample_keys = []
    for entry in entries[:sample_size]:
        labels = list(iter_entry_labels(entry))
        if not sample_keys and labels:
            sample_keys = sorted(labels[0].keys())
        entry_targets = 0
        for obj in labels:
            category = extract_category(obj) or '<missing>'
            label_counter[category] += 1
            if object_is_target_vehicle(obj):
                target_boxes += 1
                entry_targets += 1
        if entry_targets:
            entries_with_targets += 1
    print(
        f'{split} label diagnostic:',
        'sample_entries', total_entries,
        'entries_with_target_vehicle_boxes', entries_with_targets,
        'target_vehicle_boxes', target_boxes,
        'top_categories', label_counter.most_common(12),
        'sample_label_keys', sample_keys,
    )
    return {'entries': total_entries, 'entries_with_targets': entries_with_targets, 'target_boxes': target_boxes}


def cache_entries_are_usable(entries, split, cache_path):
    if not isinstance(entries, list) or not entries:
        print(f'Ignoring {split} label cache because it is empty or not a list:', cache_path)
        return False
    diag = summarize_label_entries(entries, f'{split} cache')
    if diag['target_boxes'] == 0:
        print(f'Ignoring {split} label cache because sampled entries contain 0 target vehicle boxes:', cache_path)
        return False
    return True


def load_usable_label_cache(cache_path, split):
    if not cache_path.exists():
        return None
    try:
        entries = read_json(cache_path)
    except Exception as exc:
        print(f'Could not read {split} label cache; ignoring:', cache_path, exc)
        return None
    if cache_entries_are_usable(entries, split, cache_path):
        print(f'Loading cached {split} label entries:', cache_path)
        return entries
    return None


def read_label_entries(source, split):
    if source is None:
        return []
    source = Path(source)
    cache_path = OUT_ROOT / 'metadata' / f'{split}_label_entries_cache.json'
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    cached_entries = None if FORCE_REBUILD_YOLO_DATASET else load_usable_label_cache(cache_path, split)
    if cached_entries is not None:
        return cached_entries
    if cache_path.exists():
        cache_path.unlink()
        print(f'Removed unusable local {split} label cache:', cache_path)

    drive_cache = Path(globals().get('DRIVE_OUT_ROOT', OUT_ROOT)) / 'metadata' / f'{split}_label_entries_cache.json'
    if not FORCE_REBUILD_YOLO_DATASET and drive_cache.exists() and drive_cache != cache_path:
        print(f'Checking cached {split} label entries from Drive before copying:', drive_cache)
        drive_entries = load_usable_label_cache(drive_cache, split)
        if drive_entries is not None:
            print(f'Copying usable cached {split} label entries from Drive to local workdir:', drive_cache, '->', cache_path)
            shutil.copy2(drive_cache, cache_path)
            return drive_entries
        print(f'Drive {split} label cache is not usable for vehicle detection; rebuilding from source labels.')

    entries = []
    if source.is_dir():
        files = sorted(source.glob('*.json'))
        print(f'Reading per-image {split} label JSON directory:', source, 'files:', len(files))
        for path in tqdm(files, desc=f'Reading {split} label jsons'):
            try:
                obj = read_json(path)
                fallback_name = path.with_suffix('.jpg').name
                entry = normalize_bdd_entry(obj, fallback_name=fallback_name)
                if entry is not None:
                    entries.append(entry)
            except Exception as exc:
                print('skip label json:', path, exc)
    else:
        obj = read_json(source)
        if isinstance(obj, list):
            for item in obj:
                entry = normalize_bdd_entry(item)
                if entry is not None:
                    entries.append(entry)
        else:
            entry = normalize_bdd_entry(obj, fallback_name=source.with_suffix('.jpg').name)
            if entry is not None:
                entries.append(entry)

    if not entries:
        raise ValueError(f'No usable BDD label entries loaded from {source}')
    diag = summarize_label_entries(entries, split, sample_size=min(5000, len(entries)))
    if diag['target_boxes'] == 0:
        raise RuntimeError(
            f'{split} label source loaded {len(entries)} entries but no target vehicle boxes were found. '
            'This source is likely not the BDD100K detection/object label set, or its object schema is unsupported. '
            f'Source: {source}'
        )
    cache_path.write_text(json.dumps(entries))
    print(f'{split} label entries loaded:', len(entries), 'cache:', cache_path)
    return entries


def box2d_to_yolo(box, img_w, img_h):
    x1 = max(0.0, float(box['x1']))
    y1 = max(0.0, float(box['y1']))
    x2 = min(float(img_w), float(box['x2']))
    y2 = min(float(img_h), float(box['y2']))
    if x2 <= x1 or y2 <= y1:
        return None
    xc = ((x1 + x2) / 2.0) / img_w
    yc = ((y1 + y2) / 2.0) / img_h
    w = (x2 - x1) / img_w
    h = (y2 - y1) / img_h
    return xc, yc, w, h


IMAGE_INDEX_CACHE = {}

def make_image_lookup(df):
    lookup = {}
    if df is None or df.empty:
        return lookup
    for row in df.to_dict('records'):
        path = Path(str(row['image_path']))
        image_name = str(row.get('image_name') or path.name)
        keys = {
            image_name,
            path.name,
            path.stem,
            Path(image_name).stem,
        }
        # BDD labels may arrive either as a bare stem or as a jpg/jpeg/png filename.
        for key in list(keys):
            if key:
                stem = Path(key).stem
                keys.update({stem, f'{stem}.jpg', f'{stem}.jpeg', f'{stem}.png'})
        for key in keys:
            if key and key not in lookup:
                lookup[key] = str(path)
    return lookup

def image_index_cache_matches_root(df, root, sample_size=25):
    if df is None or df.empty or 'image_path' not in df.columns:
        return False
    root = Path(root).resolve()
    checked = 0
    for value in df['image_path'].dropna().astype(str).head(sample_size):
        checked += 1
        try:
            path = Path(value).resolve()
            if root == path or root in path.parents:
                continue
            return False
        except Exception:
            return False
    return checked > 0

def build_image_index(root, split_name):
    if root is None:
        return {}
    root = Path(root)
    cache_path = OUT_ROOT / 'metadata' / f'{split_name}_image_index.csv'
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    if cache_path.exists() and not FORCE_REBUILD_YOLO_DATASET:
        try:
            df = pd.read_csv(cache_path)
            if {'image_name', 'image_path'}.issubset(df.columns) and not df.empty:
                if image_index_cache_matches_root(df, root):
                    print(f'Loading cached {split_name} image index:', cache_path, 'rows:', len(df))
                    return make_image_lookup(df)
                print(f'Ignoring stale {split_name} image index cache for a different image root:', cache_path, 'rows:', len(df))
        except Exception as exc:
            print('Could not read image index cache; rebuilding:', exc)
    print(f'Building {split_name} image index under:', root)
    rows = []
    for ext in ('*.jpg', '*.jpeg', '*.png'):
        for path in root.rglob(ext):
            rows.append({'image_name': path.name, 'image_path': str(path)})
    df = pd.DataFrame(rows).drop_duplicates(subset=['image_name'])
    df.to_csv(cache_path, index=False)
    print(f'{split_name} image index rows:', len(df), 'cache:', cache_path)
    return make_image_lookup(df)


def ensure_image_indexes():
    if 'train' not in IMAGE_INDEX_CACHE:
        IMAGE_INDEX_CACHE['train'] = build_image_index(TRAIN_IMAGE_DIR, 'train')
    if VAL_IMAGE_DIR is not None and 'val' not in IMAGE_INDEX_CACHE:
        IMAGE_INDEX_CACHE['val'] = build_image_index(VAL_IMAGE_DIR, 'val')


def reset_image_indexes():
    IMAGE_INDEX_CACHE.clear()
    for split_name in ('train', 'val'):
        cache_path = OUT_ROOT / 'metadata' / f'{split_name}_image_index.csv'
        if cache_path.exists():
            cache_path.unlink()
            print('Removed stale image index cache:', cache_path)


def refresh_bdd_image_dirs():
    global TRAIN_IMAGE_DIR, VAL_IMAGE_DIR
    TRAIN_IMAGE_DIR = find_image_dir('train')
    VAL_IMAGE_DIR = find_image_dir('val', required=False)
    print('Refreshed image dirs after archive extraction:')
    print('TRAIN_IMAGE_DIR =', TRAIN_IMAGE_DIR)
    print('VAL_IMAGE_DIR =', VAL_IMAGE_DIR)


def find_image_path(name, source_split):
    ensure_image_indexes()
    roots_by_priority = [source_split, 'train', 'val']
    for key in roots_by_priority:
        index = IMAGE_INDEX_CACHE.get(key) or {}
        value = index.get(name)
        if value:
            return Path(value)
    return None


def compute_label_image_overlap(entries, source_split, sample_size=500):
    ensure_image_indexes()
    checked = 0
    matches = 0
    missing_examples = []
    for entry in entries:
        name = entry.get('name')
        if not name:
            continue
        checked += 1
        if find_image_path(name, source_split) is not None:
            matches += 1
        elif len(missing_examples) < 8:
            missing_examples.append(name)
        if checked >= sample_size:
            break
    return checked, matches, missing_examples


def assert_label_image_overlap(entries, source_split, sample_size=500):
    checked, matches, missing_examples = compute_label_image_overlap(entries, source_split, sample_size=sample_size)
    print(f'Label/image overlap preflight for {source_split}: matches={matches}/{checked}')
    if checked and matches == 0 and globals().get('AUTO_DOWNLOAD_MATCHING_BDD_IMAGES', False):
        print('No sampled label matched current Drive images. Downloading the matching Archive.org BDD100K image archive and retrying once.')
        maybe_extract_matching_bdd_images_archive()
        reset_image_indexes()
        refresh_bdd_image_dirs()
        checked, matches, missing_examples = compute_label_image_overlap(entries, source_split, sample_size=sample_size)
        print(f'Label/image overlap after Archive.org image extraction for {source_split}: matches={matches}/{checked}')
    if checked and matches == 0:
        raise RuntimeError(
            'No matching images were found for sampled BDD labels. '
            'The current Drive image set does not match the extracted label archive.\n'
            f'Missing examples: {missing_examples}\n'
            f'TRAIN_IMAGE_DIR: {TRAIN_IMAGE_DIR}\n'
            'Use official BDD100K 100K Images or Archive.org bdd100k_images.zip: '
            f'{MATCHING_BDD_IMAGES_DOWNLOAD_URL}'
        )


def link_or_copy_image(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    try:
        os.symlink(src, dst)
    except Exception:
        shutil.copy2(src, dst)


def convert_entries(entries, source_split, limit=None):
    image_out_dir = OUT_ROOT / 'images' / 'all'
    label_out_dir = OUT_ROOT / 'labels' / 'all'
    partial_path = OUT_ROOT / 'metadata' / f'{source_split}_conversion_rows_partial.csv'
    image_out_dir.mkdir(parents=True, exist_ok=True)
    label_out_dir.mkdir(parents=True, exist_ok=True)
    partial_path.parent.mkdir(parents=True, exist_ok=True)

    rows = []
    processed_names = set()
    if partial_path.exists() and not FORCE_REBUILD_YOLO_DATASET:
        try:
            partial_df = pd.read_csv(partial_path)
            if 'image_name' in partial_df.columns and 'split' in partial_df.columns:
                rows = partial_df.to_dict('records')
                processed_names = set(partial_df['image_name'].dropna().astype(str).tolist())
                if 'source_label_name' in partial_df.columns:
                    processed_names.update(partial_df['source_label_name'].dropna().astype(str).tolist())
                print(f'Resuming {source_split} conversion from partial checkpoint:', partial_path, 'rows:', len(rows))
            else:
                print('Ignoring incomplete partial conversion file:', partial_path, 'columns:', list(partial_df.columns))
        except Exception as exc:
            print('Could not read partial conversion file; rebuilding it:', exc)

    processed = len(rows)
    skipped_existing = 0
    skipped_no_image = 0
    skipped_no_target = 0
    new_rows_since_save = 0

    for entry in tqdm(entries, desc=f'Converting {source_split}'):
        if limit is not None and processed >= limit:
            break
        name = entry.get('name')
        if not name:
            continue
        if name in processed_names or f'{Path(name).stem}.jpg' in processed_names:
            skipped_existing += 1
            continue
        img_path = find_image_path(name, source_split)
        if img_path is None:
            skipped_no_image += 1
            continue
        image_file_name = img_path.name
        try:
            with Image.open(img_path) as im:
                img_w, img_h = im.size
        except Exception:
            skipped_no_image += 1
            continue
        attrs = entry.get('attributes') or {}
        condition = condition_from_attributes(attrs)
        yolo_lines = []
        class_counts = collections.Counter()
        for obj in iter_entry_labels(entry):
            category_raw = extract_category(obj)
            if category_raw in IGNORED_CATEGORIES:
                continue
            mapped = CATEGORY_MAPPING.get(category_raw)
            if mapped is None or mapped not in CLASS_TO_ID:
                continue
            box = extract_box2d(obj)
            if not box:
                continue
            yolo_box = box2d_to_yolo(box, img_w, img_h)
            if yolo_box is None:
                continue
            cls_id = CLASS_TO_ID[mapped]
            yolo_lines.append(f'{cls_id} ' + ' '.join(f'{v:.6f}' for v in yolo_box))
            class_counts[mapped] += 1
        if not yolo_lines:
            skipped_no_target += 1
            continue
        dst_image = image_out_dir / image_file_name
        dst_label = label_out_dir / (Path(image_file_name).stem + '.txt')
        link_or_copy_image(img_path, dst_image)
        dst_label.write_text('\n'.join(yolo_lines) + '\n')
        split = split_for_name(image_file_name)
        row = {
            'image_name': image_file_name,
            'source_label_name': name,
            'image_path': str(dst_image),
            'label_path': str(dst_label),
            'source_split': source_split,
            'split': split,
            'condition_profile': condition,
            'weather': attrs.get('weather', 'undefined'),
            'timeofday': attrs.get('timeofday', 'undefined'),
            'scene': attrs.get('scene', 'undefined'),
            'width': img_w,
            'height': img_h,
            'bbox_count': len(yolo_lines),
        }
        for cls in TARGET_CLASSES:
            row[f'{cls}_count'] = class_counts.get(cls, 0)
        rows.append(row)
        processed_names.add(name)
        processed_names.add(image_file_name)
        processed += 1
        new_rows_since_save += 1
        if new_rows_since_save >= 250:
            pd.DataFrame(rows).drop_duplicates(subset=['image_name']).to_csv(partial_path, index=False)
            new_rows_since_save = 0

    if rows:
        pd.DataFrame(rows).drop_duplicates(subset=['image_name']).to_csv(partial_path, index=False)
    print(
        source_split,
        'processed_rows', len(rows),
        'skipped_existing', skipped_existing,
        'skipped_no_image', skipped_no_image,
        'skipped_no_target', skipped_no_target,
        'partial_checkpoint', partial_path,
    )
    return rows

metadata_path = OUT_ROOT / 'metadata' / 'bdd100k_vehicle_metadata.csv'
metadata_path.parent.mkdir(parents=True, exist_ok=True)
REQUIRED_METADATA_COLUMNS = {
    'image_name', 'source_label_name', 'image_path', 'label_path', 'source_split', 'split',
    'condition_profile', 'bbox_count', *[f'{cls}_count' for cls in TARGET_CLASSES]
}

def load_valid_metadata_or_none(path):
    if not path.exists():
        return None
    try:
        df = pd.read_csv(path)
    except Exception as exc:
        print('Existing metadata CSV could not be read; rebuilding conversion metadata:', exc)
        return None
    missing = sorted(REQUIRED_METADATA_COLUMNS - set(df.columns))
    if missing:
        print('Existing metadata CSV is incomplete; rebuilding conversion metadata.')
        print('Missing columns:', missing)
        print('Existing columns:', list(df.columns))
        return None
    if df.empty:
        print('Existing metadata CSV is empty; rebuilding conversion metadata.')
        return None
    return df

metadata_df = None
if not FORCE_REBUILD_YOLO_DATASET:
    metadata_df = load_valid_metadata_or_none(metadata_path)

if metadata_df is not None:
    print('Converted YOLO dataset metadata already exists and is valid. Skipping label read/conversion:')
    print(' ', metadata_path)
else:
    train_entries = read_label_entries(TRAIN_LABEL, 'train')
    if VAL_LABEL is not None and VAL_IMAGE_DIR is not None:
        val_entries = read_label_entries(VAL_LABEL, 'val')
    else:
        print('VAL label/image pair is incomplete; skipping official val conversion. Train images will be hash-split into train/val/test.')
        val_entries = []

    assert_label_image_overlap(train_entries, 'train')
    if val_entries:
        assert_label_image_overlap(val_entries, 'val')

    limit_train = SMOKE_LIMIT_IMAGES
    limit_val = None if SMOKE_LIMIT_IMAGES is None else max(100, SMOKE_LIMIT_IMAGES // 5)
    rows = []
    rows.extend(convert_entries(train_entries, 'train', limit=limit_train))
    if val_entries:
        rows.extend(convert_entries(val_entries, 'val', limit=limit_val))

    if not rows:
        for stale_path in [
            metadata_path,
            OUT_ROOT / 'metadata' / 'train_conversion_rows_partial.csv',
            OUT_ROOT / 'metadata' / 'val_conversion_rows_partial.csv',
        ]:
            if stale_path.exists():
                stale_path.unlink()
                print('Removed empty/stale conversion artifact:', stale_path)
        raise RuntimeError(
            'YOLO conversion produced 0 usable rows after label/image overlap succeeded. '
            'The selected label cache/source has no supported target vehicle boxes. '
            'Check the label diagnostics printed above; use BDD100K object detection labels with car/bus/truck/motorcycle box annotations.'
        )
    metadata_df = pd.DataFrame(rows).drop_duplicates(subset=['image_name']).reset_index(drop=True)
    missing_after_conversion = sorted(REQUIRED_METADATA_COLUMNS - set(metadata_df.columns))
    if metadata_df.empty or missing_after_conversion:
        if metadata_path.exists():
            metadata_path.unlink()
            print('Removed invalid conversion metadata:', metadata_path)
        raise RuntimeError(
            'YOLO conversion metadata is invalid after conversion. '
            f'missing={missing_after_conversion}, shape={metadata_df.shape}'
        )
    metadata_df.to_csv(metadata_path, index=False)

print('metadata:', metadata_path, metadata_df.shape)
display(metadata_df.head())


Checking cached train label entries from Drive before copying: /content/drive/MyDrive/anomali-road-safety-ai/datasets/bdd100k_vehicle_yolo/metadata/train_label_entries_cache.json
train cache label diagnostic: sample_entries 1000 entries_with_target_vehicle_boxes 991 target_vehicle_boxes 10847 top_categories [('car', 10217), ('lane/single white', 3547), ('traffic sign', 3544), ('traffic light', 2662), ('lane/road curb', 1646), ('lane/crosswalk', 1575), ('person', 1215), ('area/drivable', 907), ('area/alternative', 879), ('lane/double yellow', 490), ('truck', 431), ('lane/single yellow', 259)] sample_label_keys ['attributes', 'box2d', 'category', 'id']
Loading cached train label entries: /content/drive/MyDrive/anomali-road-safety-ai/datasets/bdd100k_vehicle_yolo/metadata/train_label_entries_cache.json
Copying usable cached train label entries from Drive to local workdir: /content/drive/MyDrive/anomali-road-safety-ai/datasets/bdd100k_vehicle_yolo/metadata/train_label_entries_cache.json ->

Reading val label jsons:   0%|          | 0/10000 [00:00<?, ?it/s]

val label diagnostic: sample_entries 5000 entries_with_target_vehicle_boxes 4945 target_vehicle_boxes 54402 top_categories [('car', 51216), ('traffic sign', 17536), ('lane/single white', 17500), ('traffic light', 13746), ('lane/road curb', 7795), ('lane/crosswalk', 7680), ('person', 7070), ('area/drivable', 4557), ('area/alternative', 4311), ('lane/double yellow', 2690), ('truck', 2145), ('lane/single yellow', 1396)] sample_label_keys ['attributes', 'box2d', 'category', 'id']
val label entries loaded: 10000 cache: /content/anomali-road-safety-ai-work/datasets/bdd100k_vehicle_yolo/metadata/val_label_entries_cache.json
Building train image index under: /content/anomali-road-safety-ai-work/datasets/bdd100k/bdd100k/images/100k/train
train image index rows: 70000 cache: /content/anomali-road-safety-ai-work/datasets/bdd100k_vehicle_yolo/metadata/train_image_index.csv
Building val image index under: /content/anomali-road-safety-ai-work/datasets/bdd100k/bdd100k/images/100k/val
val image index 

Converting train:   0%|          | 0/23193 [00:00<?, ?it/s]

train processed_rows 22999 skipped_existing 0 skipped_no_image 0 skipped_no_target 194 partial_checkpoint /content/anomali-road-safety-ai-work/datasets/bdd100k_vehicle_yolo/metadata/train_conversion_rows_partial.csv


Converting val:   0%|          | 0/10000 [00:00<?, ?it/s]

val processed_rows 9905 skipped_existing 0 skipped_no_image 0 skipped_no_target 95 partial_checkpoint /content/anomali-road-safety-ai-work/datasets/bdd100k_vehicle_yolo/metadata/val_conversion_rows_partial.csv
metadata: /content/anomali-road-safety-ai-work/datasets/bdd100k_vehicle_yolo/metadata/bdd100k_vehicle_metadata.csv (32904, 17)


,image_name,source_label_name,image_path,label_path,source_split,split,condition_profile,weather,timeofday,scene,width,height,bbox_count,car_count,bus_count,truck_count,motorcycle_count
0,0000f77c-6257be58.jpg,0000f77c-6257be58,/content/anomali-road-safety-ai-work/datasets/...,/content/anomali-road-safety-ai-work/datasets/...,train,train,day_clear,clear,daytime,city street,1280,720,2,2,0,0,0
1,0000f77c-cb820c98.jpg,0000f77c-cb820c98,/content/anomali-road-safety-ai-work/datasets/...,/content/anomali-road-safety-ai-work/datasets/...,train,test,low_light_transition,clear,dawn/dusk,residential,1280,720,7,7,0,0,0
2,0001542f-5ce3cf52.jpg,0001542f-5ce3cf52,/content/anomali-road-safety-ai-work/datasets/...,/content/anomali-road-safety-ai-work/datasets/...,train,train,night_low_light,clear,night,city street,1280,720,10,9,1,0,0
3,0001542f-7c670be8.jpg,0001542f-7c670be8,/content/anomali-road-safety-ai-work/datasets/...,/content/anomali-road-safety-ai-work/datasets/...,train,val,night_low_light,clear,night,highway,1280,720,3,3,0,0,0
4,0001542f-ec815219.jpg,0001542f-ec815219,/content/anomali-road-safety-ai-work/datasets/...,/content/anomali-road-safety-ai-work/datasets/...,train,train,night_low_light,clear,night,city street,1280,720,9,9,0,0,0


## 6. Dataset Distribution ve FTR Tabloları

FTR için sınıf, split ve condition dağılımları burada üretilir.

In [ ]:
def save_distribution_tables(metadata_df):
    required = {'split', 'image_name', 'bbox_count', 'condition_profile'} | {f'{cls}_count' for cls in TARGET_CLASSES}
    missing = sorted(required - set(metadata_df.columns))
    if missing:
        raise ValueError(
            'metadata_df is missing required columns: ' + ', '.join(missing) + '\n'
            'This means YOLO conversion metadata is incomplete or stale. '
            'Re-run the conversion cell after the notebook patch; it will validate label caches, rebuild them if needed, and recreate bdd100k_vehicle_metadata.csv.'
        )
    dist_dir = OUT_ROOT / 'metadata'
    dist_dir.mkdir(parents=True, exist_ok=True)
    split_dist = metadata_df.groupby('split').agg(images=('image_name', 'count'), boxes=('bbox_count', 'sum')).reset_index()
    condition_dist = metadata_df.groupby(['split', 'condition_profile']).agg(images=('image_name', 'count'), boxes=('bbox_count', 'sum')).reset_index()
    class_rows = []
    for split_name, group in metadata_df.groupby('split'):
        row = {'split': split_name}
        for cls in TARGET_CLASSES:
            row[cls] = int(group[f'{cls}_count'].sum())
        class_rows.append(row)
    class_dist = pd.DataFrame(class_rows)
    split_dist.to_csv(dist_dir / 'split_distribution.csv', index=False)
    condition_dist.to_csv(dist_dir / 'condition_distribution.csv', index=False)
    class_dist.to_csv(dist_dir / 'class_distribution.csv', index=False)
    return split_dist, condition_dist, class_dist

split_dist, condition_dist, class_dist = save_distribution_tables(metadata_df)
print('Split distribution')
display(split_dist)
print('Condition distribution')
display(condition_dist)
print('Class distribution')
display(class_dist)


Split distribution


,split,images,boxes
0,test,4931,54026
1,train,23055,251753
2,val,4918,54124


Condition distribution


,split,condition_profile,images,boxes
0,test,adverse_other,224,2448
1,test,day_clear,2232,26999
2,test,fog_low_visibility,11,131
3,test,low_light_transition,325,3940
4,test,night_low_light,1756,16781
5,test,rain,360,3462
6,test,tunnel_or_parking_dark,22,263
7,test,unknown,1,2
8,train,adverse_other,974,10517
9,train,day_clear,10300,123866


Class distribution


,split,car,bus,truck,motorcycle
0,test,50872,812,2120,222
1,train,236846,3857,10011,1039
2,val,51058,805,2074,187


## 7. Profile Dataset YAML Üretimi

Her profil için ayrı `train.txt`, `val.txt`, `test.txt` ve `data.yaml` üretilir. General tüm koşulları kullanır; specialist profiller yalnız ilgili condition subsetini kullanır.

In [ ]:
def write_list(path, values):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text('\n'.join(map(str, values)) + ('\n' if values else ''))


def build_profile_dataset(profile, metadata_df):
    profile_dir = OUT_ROOT / 'profiles' / profile
    profile_dir.mkdir(parents=True, exist_ok=True)
    profile_df = metadata_df[metadata_df['condition_profile'].apply(lambda c: profile_accepts_condition(profile, c))].copy()
    counts = {}
    paths = {}
    for split_name in ['train', 'val', 'test']:
        split_paths = profile_df.loc[profile_df['split'] == split_name, 'image_path'].tolist()
        list_path = profile_dir / f'{split_name}.txt'
        write_list(list_path, split_paths)
        counts[split_name] = len(split_paths)
        paths[split_name] = str(list_path)
    data_yaml = {
        'path': str(OUT_ROOT),
        'train': paths['train'],
        'val': paths['val'],
        'test': paths['test'],
        'names': {idx: name for idx, name in enumerate(TARGET_CLASSES)},
    }
    data_yaml_path = profile_dir / 'data.yaml'
    data_yaml_path.write_text(yaml.safe_dump(data_yaml, sort_keys=False, allow_unicode=True))
    summary = {'profile': profile, 'data_yaml': str(data_yaml_path), **counts}
    return summary, profile_df

profile_summaries = []
profile_frames = {}
for profile in PROFILE_CONDITIONS:
    summary, frame = build_profile_dataset(profile, metadata_df)
    profile_summaries.append(summary)
    profile_frames[profile] = frame

profile_summary_df = pd.DataFrame(profile_summaries)
profile_summary_path = OUT_ROOT / 'metadata' / 'profile_dataset_summary.csv'
profile_summary_df.to_csv(profile_summary_path, index=False)
print('Profile dataset summary:', profile_summary_path)
display(profile_summary_df)

Profile dataset summary: /content/anomali-road-safety-ai-work/datasets/bdd100k_vehicle_yolo/metadata/profile_dataset_summary.csv


,profile,data_yaml,train,val,test
0,general,/content/anomali-road-safety-ai-work/datasets/...,23055,4918,4931
1,night_low_light,/content/anomali-road-safety-ai-work/datasets/...,10116,2220,2103
2,rain,/content/anomali-road-safety-ai-work/datasets/...,1625,335,360
3,fog_low_visibility,/content/anomali-road-safety-ai-work/datasets/...,28,3,11


## 8. Eğitim ve Metrik Yardımcıları

Aşağıdaki fonksiyonlar pretrained baseline validation, fine-tune, condition breakdown, ONNX export ve CSV/JSON kayıtlarını yönetir.

In [ ]:
from ultralytics import YOLO


def safe_float(value):
    try:
        return float(value)
    except Exception:
        return None


def metrics_to_row(metrics, *, experiment_id, profile, stage, weights, split_name, data_yaml):
    box = metrics.box
    row = {
        'experiment_id': experiment_id,
        'profile': profile,
        'stage': stage,
        'weights': str(weights),
        'split': split_name,
        'data_yaml': str(data_yaml),
        'map50': safe_float(box.map50),
        'map5095': safe_float(box.map),
        'precision_mean': safe_float(box.mp),
        'recall_mean': safe_float(box.mr),
        'fitness': safe_float(getattr(metrics, 'fitness', None)),
    }
    try:
        for cls_id, cls_name in enumerate(TARGET_CLASSES):
            row[f'ap50_{cls_name}'] = safe_float(box.ap50[cls_id]) if len(box.ap50) > cls_id else None
            row[f'ap5095_{cls_name}'] = safe_float(box.ap[cls_id]) if len(box.ap) > cls_id else None
    except Exception as exc:
        print('Class AP extraction skipped:', exc)
    return row


def yolo_runtime_kwargs(for_train=True):
    kwargs = {
        'device': DEVICE,
        'workers': WORKERS,
    }
    if for_train:
        kwargs.update({
            'amp': AMP,
            'cache': CACHE_MODE,
            'deterministic': DETERMINISTIC,
            'seed': RANDOM_SEED,
            'plots': PLOTS,
            'save_period': SAVE_PERIOD,
        })
    return kwargs


def validate_weights(weights, data_yaml, *, experiment_id, profile, stage, split_name='val', name_suffix='val'):
    model = YOLO(str(weights))
    run_name = f'{experiment_id}-{stage}-{name_suffix}'
    metrics = model.val(
        data=str(data_yaml),
        split=split_name,
        imgsz=IMGSZ,
        batch=VAL_BATCH,
        project=str(RUN_ROOT / 'val'),
        name=run_name,
        exist_ok=True,
        verbose=False,
        **yolo_runtime_kwargs(for_train=False),
    )
    return metrics_to_row(metrics, experiment_id=experiment_id, profile=profile, stage=stage, weights=weights, split_name=split_name, data_yaml=data_yaml)


def find_best_checkpoint(save_dir):
    save_dir = Path(save_dir)
    candidates = list(save_dir.rglob('weights/best.pt')) + list(save_dir.rglob('best.pt'))
    if not candidates:
        raise FileNotFoundError('best.pt not found under ' + str(save_dir))
    return candidates[0]


def train_experiment(exp, data_yaml, start_weights):
    experiment_id = exp['experiment_id']
    run_dir = RUN_ROOT / 'train' / experiment_id
    best_pt_existing = run_dir / 'weights' / 'best.pt'
    last_pt_existing = run_dir / 'weights' / 'last.pt'

    if best_pt_existing.exists() and not FORCE_RETRAIN_MODELS:
        print('Existing best.pt found. Skipping training for', experiment_id)
        print(' ', best_pt_existing)
        return best_pt_existing

    if RESUME_INTERRUPTED_TRAINING and last_pt_existing.exists() and not FORCE_RETRAIN_MODELS:
        print('Resuming interrupted training for', experiment_id)
        print(' ', last_pt_existing)
        model = YOLO(str(last_pt_existing))
        results = model.train(resume=True, **yolo_runtime_kwargs(for_train=True))
    else:
        model = YOLO(str(start_weights))
        results = model.train(
            data=str(data_yaml),
            epochs=int(exp['epochs']),
            imgsz=IMGSZ,
            batch=TRAIN_BATCH,
            patience=PATIENCE,
            project=str(RUN_ROOT / 'train'),
            name=experiment_id,
            exist_ok=True,
            verbose=True,
            **yolo_runtime_kwargs(for_train=True),
        )
    save_dir = getattr(results, 'save_dir', None) or run_dir
    best_pt = find_best_checkpoint(save_dir)
    return Path(best_pt)

def maybe_export_onnx(best_pt, experiment_id):
    if not RUN_EXPORT_ONNX:
        return None
    model = YOLO(str(best_pt))
    exported = model.export(format='onnx', imgsz=IMGSZ, dynamic=False, simplify=True)
    print('ONNX exported for', experiment_id, exported)
    return str(exported)


def data_yaml_for_profile(profile):
    return OUT_ROOT / 'profiles' / profile / 'data.yaml'


def profile_counts(profile):
    row = profile_summary_df[profile_summary_df['profile'] == profile].iloc[0].to_dict()
    return {k: int(row[k]) for k in ['train', 'val', 'test']}

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


## 9. End-to-End Experiment Loop

Bu döngü şunları üretir:

- pretrained/start baseline validation,
- fine-tuned validation/test,
- condition breakdown,
- specialist vs general comparison,
- `.pt` checkpoint path kayıtları,
- opsiyonel ONNX export.

In [ ]:
overall_rows = []
condition_rows = []
delta_rows = []
specialist_comparison_rows = []
model_registry = []
best_general_pt = None

for exp in EXPERIMENTS:
    experiment_id = exp['experiment_id']
    profile = exp['profile']
    if not exp.get('enabled', False):
        print('Skipping disabled experiment:', experiment_id)
        continue
    counts = profile_counts(profile)
    print('\n===', experiment_id, profile, counts, '===')
    if counts['train'] < MIN_TRAIN_IMAGES_FOR_EXPERIMENT or counts['val'] < MIN_VAL_IMAGES_FOR_EXPERIMENT:
        print('Skipping because subset is too small:', counts)
        model_registry.append({
            'experiment_id': experiment_id,
            'profile': profile,
            'status': 'skipped_subset_too_small',
            **counts,
        })
        continue

    data_yaml = data_yaml_for_profile(profile)
    start_weights = exp['model']
    baseline_stage = 'pretrained_baseline'
    if exp.get('start_from_general') and best_general_pt is not None:
        start_weights = str(best_general_pt)
        baseline_stage = 'best_general_baseline'

    baseline_val = validate_weights(start_weights, data_yaml, experiment_id=experiment_id, profile=profile, stage=baseline_stage, split_name='val', name_suffix='val')
    overall_rows.append(baseline_val)
    if counts['test'] >= MIN_VAL_IMAGES_FOR_EXPERIMENT:
        baseline_test = validate_weights(start_weights, data_yaml, experiment_id=experiment_id, profile=profile, stage=baseline_stage, split_name='test', name_suffix='test')
        overall_rows.append(baseline_test)

    best_pt = train_experiment(exp, data_yaml, start_weights)
    if profile == 'general':
        best_general_pt = best_pt

    finetuned_val = validate_weights(best_pt, data_yaml, experiment_id=experiment_id, profile=profile, stage='finetuned', split_name='val', name_suffix='val')
    overall_rows.append(finetuned_val)
    if counts['test'] >= MIN_VAL_IMAGES_FOR_EXPERIMENT:
        finetuned_test = validate_weights(best_pt, data_yaml, experiment_id=experiment_id, profile=profile, stage='finetuned', split_name='test', name_suffix='test')
        overall_rows.append(finetuned_test)

    onnx_path = maybe_export_onnx(best_pt, experiment_id)
    model_registry.append({
        'experiment_id': experiment_id,
        'profile': profile,
        'status': 'trained',
        'best_pt': str(best_pt),
        'onnx': onnx_path,
        **counts,
    })

    for split_name in ['val', 'test']:
        base = next((r for r in overall_rows if r['experiment_id'] == experiment_id and r['stage'] == baseline_stage and r['split'] == split_name), None)
        tuned = next((r for r in overall_rows if r['experiment_id'] == experiment_id and r['stage'] == 'finetuned' and r['split'] == split_name), None)
        if base and tuned:
            delta_rows.append({
                'experiment_id': experiment_id,
                'profile': profile,
                'split': split_name,
                'baseline_stage': baseline_stage,
                'delta_map50': (tuned['map50'] or 0) - (base['map50'] or 0),
                'delta_map5095': (tuned['map5095'] or 0) - (base['map5095'] or 0),
                'delta_precision_mean': (tuned['precision_mean'] or 0) - (base['precision_mean'] or 0),
                'delta_recall_mean': (tuned['recall_mean'] or 0) - (base['recall_mean'] or 0),
            })

    # General model condition breakdown. Specialist models are evaluated on all available profile YAMLs too.
    for eval_profile in PROFILE_CONDITIONS.keys():
        eval_counts = profile_counts(eval_profile)
        if eval_counts['val'] < MIN_VAL_IMAGES_FOR_EXPERIMENT:
            continue
        eval_yaml = data_yaml_for_profile(eval_profile)
        row = validate_weights(best_pt, eval_yaml, experiment_id=experiment_id, profile=eval_profile, stage='condition_breakdown', split_name='val', name_suffix=f'{eval_profile}-val')
        row['trained_profile'] = profile
        condition_rows.append(row)
        if eval_counts['test'] >= MIN_VAL_IMAGES_FOR_EXPERIMENT:
            row_test = validate_weights(best_pt, eval_yaml, experiment_id=experiment_id, profile=eval_profile, stage='condition_breakdown', split_name='test', name_suffix=f'{eval_profile}-test')
            row_test['trained_profile'] = profile
            condition_rows.append(row_test)

    if profile != 'general' and best_general_pt is not None:
        own_yaml = data_yaml_for_profile(profile)
        general_row = validate_weights(best_general_pt, own_yaml, experiment_id=experiment_id, profile=profile, stage='general_comparison', split_name='test', name_suffix='general-comparison-test')
        specialist_row = validate_weights(best_pt, own_yaml, experiment_id=experiment_id, profile=profile, stage='specialist_comparison', split_name='test', name_suffix='specialist-comparison-test')
        specialist_comparison_rows.append({
            'experiment_id': experiment_id,
            'profile': profile,
            'split': 'test',
            'general_map5095': general_row['map5095'],
            'specialist_map5095': specialist_row['map5095'],
            'delta_map5095': (specialist_row['map5095'] or 0) - (general_row['map5095'] or 0),
            'general_recall': general_row['recall_mean'],
            'specialist_recall': specialist_row['recall_mean'],
            'delta_recall': (specialist_row['recall_mean'] or 0) - (general_row['recall_mean'] or 0),
        })

print('Finished experiments')


=== VD-EXP-002-GENERAL-YOLO11N general {'train': 23055, 'val': 4918, 'test': 4931} ===
Ultralytics 8.4.67 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
YOLO11n summary (fused): 100 layers, 2,616,248 parameters, 0 gradients, 6.5 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1493.1±441.8 MB/s, size: 56.9 KB)
val: Scanning /content/anomali-road-safety-ai-work/datasets/bdd100k_vehicle_yolo/labels/all... 4918 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 4918/4918 1.5Kit/s 3.2s
val: New cache created: /content/anomali-road-safety-ai-work/datasets/bdd100k_vehicle_yolo/labels/all.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 77/77 3.3it/s 23.3s
                   all       4918      54124      0.086     0.0965      0.039     0.0182
Speed: 0.4ms preprocess, 0.6ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /content/drive/MyDrive/anomali-road-safety-ai/run

## 10. Sonuç CSV/JSON Kayıtları

In [ ]:
overall_df = pd.DataFrame(overall_rows)
condition_df = pd.DataFrame(condition_rows)
delta_df = pd.DataFrame(delta_rows)
specialist_df = pd.DataFrame(specialist_comparison_rows)
registry_df = pd.DataFrame(model_registry)

outputs = {
    'overall_metrics': overall_df,
    'condition_breakdown_metrics': condition_df,
    'baseline_vs_finetuned_delta': delta_df,
    'specialist_vs_general_comparison': specialist_df,
    'model_registry': registry_df,
}

for name, df in outputs.items():
    csv_path = SUMMARY_ROOT / f'{name}.csv'
    json_path = SUMMARY_ROOT / f'{name}.json'
    df.to_csv(csv_path, index=False)
    df.to_json(json_path, orient='records', indent=2, force_ascii=False)
    print(name, '->', csv_path, df.shape)
    if not df.empty:
        display(df.head(20))

overall_metrics -> /content/drive/MyDrive/anomali-road-safety-ai/runs/vehicle_detection/VD-EXP-002/summary/overall_metrics.csv (12, 19)


,experiment_id,profile,stage,weights,split,data_yaml,map50,map5095,precision_mean,recall_mean,fitness,ap50_car,ap5095_car,ap50_bus,ap5095_bus,ap50_truck,ap5095_truck,ap50_motorcycle,ap5095_motorcycle
0,VD-EXP-002-GENERAL-YOLO11N,general,pretrained_baseline,yolo11n.pt,val,/content/anomali-road-safety-ai-work/datasets/...,0.039033,0.018228,0.086027,0.096450,0.018228,0.006462,0.001778,0.000000e+00,0.000000e+00,0.006417,0.003001,0.143251,0.068132
1,VD-EXP-002-GENERAL-YOLO11N,general,pretrained_baseline,yolo11n.pt,test,/content/anomali-road-safety-ai-work/datasets/...,0.041797,0.020412,0.091624,0.102383,0.020412,0.006392,0.001803,4.742033e-07,9.484067e-08,0.007258,0.003551,0.153539,0.076295
2,VD-EXP-002-GENERAL-YOLO11N,general,finetuned,/content/drive/MyDrive/anomali-road-safety-ai/...,val,/content/anomali-road-safety-ai-work/datasets/...,0.488905,0.322091,0.647045,0.425098,0.322091,0.725385,0.436591,4.959652e-01,3.785548e-01,0.525341,0.373002,0.208930,0.100217
3,VD-EXP-002-GENERAL-YOLO11N,general,finetuned,/content/drive/MyDrive/anomali-road-safety-ai/...,test,/content/anomali-road-safety-ai-work/datasets/...,0.506167,0.332283,0.632970,0.455739,0.332283,0.738859,0.445283,5.139289e-01,3.992189e-01,0.517843,0.370224,0.254037,0.114408
4,VD-EXP-003-NIGHT-YOLO11N,night_low_light,best_general_baseline,/content/drive/MyDrive/anomali-road-safety-ai/...,val,/content/anomali-road-safety-ai-work/datasets/...,0.495666,0.317889,0.605444,0.472969,0.317889,0.714865,0.409332,4.897190e-01,3.806467e-01,0.535658,0.376769,0.242422,0.104808
5,VD-EXP-003-NIGHT-YOLO11N,night_low_light,best_general_baseline,/content/drive/MyDrive/anomali-road-safety-ai/...,test,/content/anomali-road-safety-ai-work/datasets/...,0.496828,0.318208,0.598343,0.460183,0.318208,0.730319,0.417107,4.615779e-01,3.640027e-01,0.498678,0.354911,0.296739,0.136813
6,VD-EXP-003-NIGHT-YOLO11N,night_low_light,finetuned,/content/drive/MyDrive/anomali-road-safety-ai/...,val,/content/anomali-road-safety-ai-work/datasets/...,0.457136,0.289840,0.607022,0.429399,0.289840,0.706685,0.402843,4.520155e-01,3.363076e-01,0.500281,0.346616,0.169564,0.073593
7,VD-EXP-003-NIGHT-YOLO11N,night_low_light,finetuned,/content/drive/MyDrive/anomali-road-safety-ai/...,test,/content/anomali-road-safety-ai-work/datasets/...,0.478761,0.299556,0.572349,0.486511,0.299556,0.723664,0.410958,4.317845e-01,3.312249e-01,0.476408,0.337255,0.283187,0.118787
8,VD-EXP-004-RAIN-YOLO11N,rain,best_general_baseline,/content/drive/MyDrive/anomali-road-safety-ai/...,val,/content/anomali-road-safety-ai-work/datasets/...,0.531307,0.358324,0.714387,0.462664,0.358324,0.752477,0.454348,5.189053e-01,4.028178e-01,0.590640,0.428866,0.263205,0.147265
9,VD-EXP-004-RAIN-YOLO11N,rain,best_general_baseline,/content/drive/MyDrive/anomali-road-safety-ai/...,test,/content/anomali-road-safety-ai-work/datasets/...,0.473641,0.318531,0.640104,0.385243,0.318531,0.724160,0.440587,4.787945e-01,3.771782e-01,0.549210,0.399893,0.142400,0.056466


condition_breakdown_metrics -> /content/drive/MyDrive/anomali-road-safety-ai/runs/vehicle_detection/VD-EXP-002/summary/condition_breakdown_metrics.csv (18, 20)


,experiment_id,profile,stage,weights,split,data_yaml,map50,map5095,precision_mean,recall_mean,fitness,ap50_car,ap5095_car,ap50_bus,ap5095_bus,ap50_truck,ap5095_truck,ap50_motorcycle,ap5095_motorcycle,trained_profile
0,VD-EXP-002-GENERAL-YOLO11N,general,condition_breakdown,/content/drive/MyDrive/anomali-road-safety-ai/...,val,/content/anomali-road-safety-ai-work/datasets/...,0.488905,0.322091,0.647045,0.425098,0.322091,0.725385,0.436591,0.495965,0.378555,0.525341,0.373002,0.208930,0.100217,general
1,VD-EXP-002-GENERAL-YOLO11N,general,condition_breakdown,/content/drive/MyDrive/anomali-road-safety-ai/...,test,/content/anomali-road-safety-ai-work/datasets/...,0.506167,0.332283,0.632970,0.455739,0.332283,0.738859,0.445283,0.513929,0.399219,0.517843,0.370224,0.254037,0.114408,general
2,VD-EXP-002-GENERAL-YOLO11N,night_low_light,condition_breakdown,/content/drive/MyDrive/anomali-road-safety-ai/...,val,/content/anomali-road-safety-ai-work/datasets/...,0.495666,0.317889,0.605444,0.472969,0.317889,0.714865,0.409332,0.489719,0.380647,0.535658,0.376769,0.242422,0.104808,general
3,VD-EXP-002-GENERAL-YOLO11N,night_low_light,condition_breakdown,/content/drive/MyDrive/anomali-road-safety-ai/...,test,/content/anomali-road-safety-ai-work/datasets/...,0.496828,0.318208,0.598343,0.460183,0.318208,0.730319,0.417107,0.461578,0.364003,0.498678,0.354911,0.296739,0.136813,general
4,VD-EXP-002-GENERAL-YOLO11N,rain,condition_breakdown,/content/drive/MyDrive/anomali-road-safety-ai/...,val,/content/anomali-road-safety-ai-work/datasets/...,0.531307,0.358324,0.714387,0.462664,0.358324,0.752477,0.454348,0.518905,0.402818,0.590640,0.428866,0.263205,0.147265,general
5,VD-EXP-002-GENERAL-YOLO11N,rain,condition_breakdown,/content/drive/MyDrive/anomali-road-safety-ai/...,test,/content/anomali-road-safety-ai-work/datasets/...,0.473641,0.318531,0.640104,0.385243,0.318531,0.724160,0.440587,0.478795,0.377178,0.549210,0.399893,0.142400,0.056466,general
6,VD-EXP-003-NIGHT-YOLO11N,general,condition_breakdown,/content/drive/MyDrive/anomali-road-safety-ai/...,val,/content/anomali-road-safety-ai-work/datasets/...,0.463085,0.302639,0.631086,0.421752,0.302639,0.716309,0.429013,0.467880,0.348244,0.488819,0.344112,0.179333,0.089189,night_low_light
7,VD-EXP-003-NIGHT-YOLO11N,general,condition_breakdown,/content/drive/MyDrive/anomali-road-safety-ai/...,test,/content/anomali-road-safety-ai-work/datasets/...,0.484960,0.314684,0.651380,0.436288,0.314684,0.730430,0.437484,0.482942,0.364801,0.490639,0.349524,0.235830,0.106925,night_low_light
8,VD-EXP-003-NIGHT-YOLO11N,night_low_light,condition_breakdown,/content/drive/MyDrive/anomali-road-safety-ai/...,val,/content/anomali-road-safety-ai-work/datasets/...,0.457136,0.289840,0.607022,0.429399,0.289840,0.706685,0.402843,0.452015,0.336308,0.500281,0.346616,0.169564,0.073593,night_low_light
9,VD-EXP-003-NIGHT-YOLO11N,night_low_light,condition_breakdown,/content/drive/MyDrive/anomali-road-safety-ai/...,test,/content/anomali-road-safety-ai-work/datasets/...,0.478761,0.299556,0.572349,0.486511,0.299556,0.723664,0.410958,0.431784,0.331225,0.476408,0.337255,0.283187,0.118787,night_low_light


baseline_vs_finetuned_delta -> /content/drive/MyDrive/anomali-road-safety-ai/runs/vehicle_detection/VD-EXP-002/summary/baseline_vs_finetuned_delta.csv (6, 8)


,experiment_id,profile,split,baseline_stage,delta_map50,delta_map5095,delta_precision_mean,delta_recall_mean
0,VD-EXP-002-GENERAL-YOLO11N,general,val,pretrained_baseline,0.449873,0.303864,0.561018,0.328648
1,VD-EXP-002-GENERAL-YOLO11N,general,test,pretrained_baseline,0.464370,0.311871,0.541346,0.353356
2,VD-EXP-003-NIGHT-YOLO11N,night_low_light,val,best_general_baseline,-0.038530,-0.028049,0.001578,-0.043570
3,VD-EXP-003-NIGHT-YOLO11N,night_low_light,test,best_general_baseline,-0.018068,-0.018652,-0.025994,0.026328
4,VD-EXP-004-RAIN-YOLO11N,rain,val,best_general_baseline,-0.005055,-0.012018,0.027563,0.001576
5,VD-EXP-004-RAIN-YOLO11N,rain,test,best_general_baseline,-0.005145,-0.003351,0.029487,0.015183


specialist_vs_general_comparison -> /content/drive/MyDrive/anomali-road-safety-ai/runs/vehicle_detection/VD-EXP-002/summary/specialist_vs_general_comparison.csv (2, 9)


,experiment_id,profile,split,general_map5095,specialist_map5095,delta_map5095,general_recall,specialist_recall,delta_recall
0,VD-EXP-003-NIGHT-YOLO11N,night_low_light,test,0.318208,0.299556,-0.018652,0.460183,0.486511,0.026328
1,VD-EXP-004-RAIN-YOLO11N,rain,test,0.318531,0.315180,-0.003351,0.385243,0.400426,0.015183


model_registry -> /content/drive/MyDrive/anomali-road-safety-ai/runs/vehicle_detection/VD-EXP-002/summary/model_registry.csv (4, 8)


,experiment_id,profile,status,best_pt,onnx,train,val,test
0,VD-EXP-002-GENERAL-YOLO11N,general,trained,/content/drive/MyDrive/anomali-road-safety-ai/...,NaN,23055,4918,4931
1,VD-EXP-003-NIGHT-YOLO11N,night_low_light,trained,/content/drive/MyDrive/anomali-road-safety-ai/...,NaN,10116,2220,2103
2,VD-EXP-004-RAIN-YOLO11N,rain,trained,/content/drive/MyDrive/anomali-road-safety-ai/...,NaN,1625,335,360
3,VD-EXP-005-FOG-YOLO11N,fog_low_visibility,skipped_subset_too_small,NaN,NaN,28,3,11


## 11. FTR Özet Raporu Üretimi

Bu hücre Drive altında Markdown özet üretir. Repo'ya yalnız küçük özetler aktarılmalıdır; checkpoint ve görseller Git'e eklenmez.

In [ ]:
def df_to_md(df, max_rows=20):
    if df.empty:
        return '_No rows._'
    return df.head(max_rows).to_markdown(index=False)

report_path = SUMMARY_ROOT / 'VD-EXP-002-ftr-vehicle-detection-summary.md'
report = f"""# VD-EXP-002 FTR Vehicle Detection Summary

## Scope

- Model family: YOLO11n
- Dataset: BDD100K vehicle subset
- Target classes: {', '.join(TARGET_CLASSES)}
- Main output: `.pt` checkpoint
- Optional output: ONNX export = {RUN_EXPORT_ONNX}
- Smoke conversion limit: {SMOKE_LIMIT_IMAGES}

## Dataset Split Distribution

{df_to_md(split_dist)}

## Condition Distribution

{df_to_md(condition_dist)}

## Class Distribution

{df_to_md(class_dist)}

## Profile Dataset Summary

{df_to_md(profile_summary_df)}

## Overall Metrics

{df_to_md(overall_df)}

## Condition Breakdown Metrics

{df_to_md(condition_df)}

## Baseline vs Fine-Tuned Delta

{df_to_md(delta_df)}

## Specialist vs General Comparison

{df_to_md(specialist_df)}

## Model Registry

{df_to_md(registry_df)}

## Interpretation Notes

- Local dark videos are not training data; use them only as manual qualitative smoke tests.
- Specialist models should be promoted only if they improve the target condition over the general detector and remain within runtime budget.
- This notebook does not claim legal evidence or final field performance.
"""
report_path.write_text(report)
print('FTR summary report:', report_path)
print(report[:2000])

FTR summary report: /content/drive/MyDrive/anomali-road-safety-ai/runs/vehicle_detection/VD-EXP-002/summary/VD-EXP-002-ftr-vehicle-detection-summary.md
# VD-EXP-002 FTR Vehicle Detection Summary

## Scope

- Model family: YOLO11n
- Dataset: BDD100K vehicle subset
- Target classes: car, bus, truck, motorcycle
- Main output: `.pt` checkpoint
- Optional output: ONNX export = False
- Smoke conversion limit: None

## Dataset Split Distribution

| split   |   images |   boxes |
|:--------|---------:|--------:|
| test    |     4931 |   54026 |
| train   |    23055 |  251753 |
| val     |     4918 |   54124 |

## Condition Distribution

| split   | condition_profile      |   images |   boxes |
|:--------|:-----------------------|---------:|--------:|
| test    | adverse_other          |      224 |    2448 |
| test    | day_clear              |     2232 |   26999 |
| test    | fog_low_visibility     |       11 |     131 |
| test    | low_light_transition   |      325 |    3940 |
| test    | nig

## 12. Opsiyonel Local Dark Video Smoke Test

Bu bölüm eğitim verisi değildir. Drive altında `LOCAL_SMOKE_VIDEO_DIR` içine koyulan videolar üzerinde qualitative inference üretir.

In [ ]:
if RUN_LOCAL_SMOKE:
    smoke_videos = []
    for ext in ['*.mp4', '*.mov', '*.avi', '*.mkv']:
        smoke_videos.extend(LOCAL_SMOKE_VIDEO_DIR.glob(ext))
    if not smoke_videos:
        print('No smoke videos found under', LOCAL_SMOKE_VIDEO_DIR)
    else:
        smoke_weights = None
        trained = registry_df[registry_df['status'] == 'trained'] if not registry_df.empty else pd.DataFrame()
        general_rows = trained[trained['profile'] == 'general'] if not trained.empty else pd.DataFrame()
        if not general_rows.empty:
            smoke_weights = general_rows.iloc[0]['best_pt']
        elif not trained.empty:
            smoke_weights = trained.iloc[0]['best_pt']
        if smoke_weights is None:
            print('No trained model found for smoke test.')
        else:
            smoke_model = YOLO(str(smoke_weights))
            for video in smoke_videos:
                print('Predicting:', video)
                smoke_model.predict(
                    source=str(video),
                    imgsz=IMGSZ,
                    conf=0.25,
                    project=str(RUN_ROOT / 'smoke_predictions'),
                    name=video.stem,
                    save=True,
                    exist_ok=True,
                )
else:
    print('RUN_LOCAL_SMOKE is False. Skipping local dark video smoke test.')

RUN_LOCAL_SMOKE is False. Skipping local dark video smoke test.


## 13. Repo'ya Aktarım Checklist

Notebook tamamlandıktan sonra repo'ya sadece küçük metin/CSV/JSON özetleri aktarılmalıdır.

Aktarılabilir:

- `overall_metrics.csv`
- `condition_breakdown_metrics.csv`
- `baseline_vs_finetuned_delta.csv`
- `specialist_vs_general_comparison.csv`
- `model_registry.csv` içindeki checkpoint path özeti
- `VD-EXP-002-ftr-vehicle-detection-summary.md`

Aktarılmayacak:

- raw BDD100K verisi
- `.pt`, `.onnx`, `.engine`, `.mlpackage`
- run görselleri/videoları
- kişisel veri içerebilecek frame/crop/evidence çıktıları